# 🌀 25 · Variational inference and variational autoencoders
### From EM to a mini-scVI: KL divergence, the evidence lower bound, mean-field and stochastic variational inference, the reparameterisation trick, and a negative-binomial VAE that learns cell states from raw counts and corrects batch effects

**BiologyToML** · Level 2 · Research frontier · Part F — Modern machine-learning building blocks · Module 25 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/25_variational_inference_vae.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, PyTorch (CPU), pandas, Matplotlib, scikit-learn |
| **Prerequisites** | Level 1 (especially Modules 05, 13, 21 and 22); Module 24 (negative-binomial likelihood) is helpful — the pieces we need are re-explained here |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [05 · Likelihood, MLE and MAP](05_likelihood_mle_map.ipynb), [13 · Mixture models and EM](13_mixture_models_em.ipynb), [14 · PCA](14_pca_dimensionality_reduction.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [24 · GLMs for count data](24_glm_count_data.ipynb) |
| **Research frontier covered** | variational inference, deep generative models of single cells (scVI-style VAEs), batch-aware latent spaces — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED**, so that every answer can be checked against the truth: a gene's log-expression in 20 cells
> (true mean and precision known), case–control genotypes from `simulate_genotypes` with a logistic disease model
> (true SNP effects known, one null SNP in strong linkage disequilibrium with a causal one), and PBMC-like single-cell
> UMI counts from `simulate_scrna` (true cell type, donor, batch and stimulation of every cell; true gene dispersions).
> Real single-cell data add ambient RNA, doublets, many more genes and cells, and batch effects that are not a simple
> per-gene shift — see 🧭 *From toy to real data*. The networks are deliberately tiny so that everything trains on a CPU
> in seconds.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why the posterior $p(z\mid x)$ of most latent-variable models is intractable, and **compare** the three answers — exact EM, MCMC and variational inference (VI) — in accuracy, speed and scalability.
2. **Compute** Kullback–Leibler divergences and **demonstrate** that minimising $\mathrm{KL}(q\,\|\,p)$ is mode-seeking while minimising $\mathrm{KL}(p\,\|\,q)$ is mass-covering.
3. **Derive** the identity $\log p(x)=\mathrm{ELBO}(q)+\mathrm{KL}(q\,\|\,p(z\mid x))$ and **recognise** EM as coordinate ascent on the ELBO.
4. **Derive and implement** coordinate-ascent mean-field VI (CAVI) for a Normal–Gamma model and **show** against the exact posterior that it gets the means right and the spread and dependence wrong.
5. **Implement** the score-function and reparameterisation gradient estimators, **measure** their variance, and **fit** Bayesian logistic regression by stochastic VI in PyTorch, checked against Metropolis–Hastings.
6. **Build** a variational autoencoder with a Gaussian decoder and a negative-binomial decoder (a mini-scVI) and **evaluate** its latent space and its posterior predictive distribution against the simulation's ground truth.
7. **Condition** the VAE on batch, **quantify** batch mixing and cell-type conservation, and **decode** counterfactual cells ("what would this cell look like in the other batch?").
8. **Diagnose** posterior collapse, the role of the KL weight and latent dimension, and why latent axes are not identifiable.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Why approximate inference? | EM vs MCMC vs VI; KL divergence; mode-seeking vs mass-covering | 20 min |
| 2 · The evidence lower bound | derive the ELBO; EM as coordinate ascent | 15 min |
| 3 · Mean-field VI by hand (CAVI) | Normal–Gamma posterior of one gene; exact vs mean-field | 20 min |
| 4 · Stochastic VI and the reparameterisation trick | gradient estimators; Bayesian logistic regression vs MCMC | 25 min |
| 5 · Amortised inference: the VAE | encoder, decoder, ELBO; Gaussian VAE and NB mini-scVI | 30 min |
| 6 · 🧪 Is the latent space meaningful? | cell-type metrics vs PCA; posterior predictive checks | 15 min |
| 7 · Conditioning on batch | batch-aware VAE, mixing metrics, counterfactual decoding | 20 min |
| 8 · Pitfalls | posterior collapse, KL weight, latent dimension, identifiability | 15 min |
| 9 · 🧭 From toy to real data | scvi-tools, atlases, what goes wrong first | 5 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

A droplet single-cell RNA-seq experiment measures, for each of thousands of cells, how many mRNA molecules of each of
~20,000 genes were captured. Biologically, those thousands of numbers are not independent: a cell's profile is largely
determined by **a few hidden factors** — its type (T cell, B cell, monocyte …), its state (resting or responding to
interferon), its position in the cell cycle. On top of that, the measurement adds **technical factors**: how many
molecules were captured from this cell (the *library size*), sampling noise in counts, and systematic differences between
the batches in which samples were processed.

So the natural model is a **latent-variable model**: a low-dimensional hidden vector $z$ (the *cell state*) generates the
counts $x$ through a probabilistic model of sequencing. You already know two such models — the Gaussian mixture
([Module 13](13_mixture_models_em.ipynb), where $z$ is a cluster label) and PCA viewed probabilistically
([Module 14](14_pca_dimensionality_reduction.ipynb), where $z$ is a vector of scores). Both have *exact* posteriors
$p(z\mid x)$. As soon as the map from $z$ to $x$ is a neural network and $x$ is a count, the posterior can no longer be
computed. **Variational inference** replaces the impossible computation by an optimisation, and the **variational
autoencoder** (VAE) learns that optimisation for all cells at once.

This is exactly what **scVI** (Lopez *et al.* 2018) does: a VAE with a negative-binomial likelihood for raw UMI counts, the
observed library size built into the decoder and the batch as an input, trained by stochastic variational inference on
mini-batches of cells — which is why it scales to atlases with millions of cells. Its latent space is used for
clustering, visualisation, batch integration and reference mapping, and the same machinery is now a workhorse across
computational biology.

In this notebook you build the whole chain from first principles, checking every step against a known truth: KL divergence
→ the evidence lower bound → mean-field VI by hand → stochastic VI with the reparameterisation trick → a VAE → a
**mini-scVI** on SIMULATED PBMC-like counts with six known cell types, donors, interferon stimulation and (in Section 7)
two batches — a design modelled on the interferon-β PBMC experiment of Kang *et al.* (2018).

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from scipy.special import gammaln
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                  # PyTorch has its own random-number generator
torch.set_num_threads(1)                 # our networks are tiny: one thread is about as fast and avoids contention
DEVICE = torch.device("cpu")             # everything here runs on a CPU in seconds
print("PyTorch", torch.__version__, "| device:", DEVICE, "| threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_scrna` generates PBMC-like UMI counts with known cell types, donors, batches and stimulation (it uses `sample_negative_binomial`, which we also call directly); `simulate_genotypes` draws SNP genotypes; `log_normalize` and `pca` are the standard preprocessing baseline; `plot_embedding` draws 2-D maps of cells; `sigmoid` is the logistic function. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_genotypes', 'sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'sigmoid', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · Why approximate inference?

> 📍 **Where we are.** In [Module 13](13_mixture_models_em.ipynb) the hidden variable was a cluster label, and the
> E-step computed its posterior exactly with Bayes' rule. In [Module 21](21_bayesian_learning_mcmc.ipynb) posteriors
> without a closed form were described by MCMC samples. This section explains why neither scales to a deep generative
> model of a million cells, and introduces the yardstick of everything that follows: the **KL divergence**.

A latent-variable model specifies a prior $p(z)$ over hidden variables and a likelihood $p(x\mid z)$ for the data. Bayes'
rule gives the posterior

$$p(z\mid x)=\frac{p(x\mid z)\,p(z)}{p(x)},\qquad p(x)=\int p(x\mid z)\,p(z)\,dz .$$

*In words:* to know which hidden states are plausible for an observed cell we must divide by $p(x)$, the probability of
the data averaged over **all** hidden states — the *evidence* or *marginal likelihood*.

For a mixture model the "integral" is a sum over $K$ labels, which is easy. For a cell whose state $z$ is a vector of 10
real numbers and whose expected counts are a neural-network function of $z$, $p(x)$ is a 10-dimensional integral of a
function with no closed form — and we need one for every cell. Three strategies exist:

| Strategy | What it does | Strengths | Weaknesses |
|---|---|---|---|
| **Exact inference** (EM's E-step, conjugate models) | computes $p(z\mid x)$ by formula | exact, fast | only for special models (mixtures, linear-Gaussian, conjugate priors) |
| **MCMC** ([Module 21](21_bayesian_learning_mcmc.ipynb)) | draws samples whose distribution converges to $p(z\mid x)$ | asymptotically exact, general | slow; a chain per data point; convergence hard to assess |
| **Variational inference** (this module) | picks the closest member $q(z)$ of a simple family by optimisation | fast, scales to millions of points with stochastic gradients, works with neural networks | biased: the answer is only as good as the family; typically **over-confident** |

To say "closest" we need a way to measure how different two distributions are.

🧮 **Math refresher — Kullback–Leibler (KL) divergence.** For two probability distributions $q$ and $p$ over the same
variable,

$$\mathrm{KL}(q\,\|\,p)=\sum_z q(z)\log\frac{q(z)}{p(z)}=\mathbb E_{q}\big[\log q(z)-\log p(z)\big]
\qquad\Big(\text{or }\int q(z)\log\tfrac{q(z)}{p(z)}\,dz\text{ for densities}\Big).$$

*In words:* average, under $q$, the log-ratio of the two probabilities. Three facts matter:

1. $\mathrm{KL}(q\,\|\,p)\ge0$, with equality only when $q=p$. (Proof: by Jensen's inequality from
   [Module 13 §5.3](13_mixture_models_em.ipynb), $-\mathrm{KL}=\mathbb E_q[\log(p/q)]\le\log\mathbb E_q[p/q]=\log\sum_z p(z)=0$.)
2. It is **not symmetric**: $\mathrm{KL}(q\,\|\,p)\ne\mathrm{KL}(p\,\|\,q)$ in general, so it is not a distance.
3. It is measured in **nats** (natural log) — the expected extra code length, in units of $\log e$, paid when the data
   really come from $q$ but the code was built for $p$ (with $\log_2$ it would be bits).

In [ ]:
# ── 1.1 KL divergence between two cell-type compositions ──
def kl_discrete(q, p):
    """KL(q ‖ p) = Σ q log(q/p) in nats for two probability vectors (terms with q = 0 contribute 0)."""
    q, p = np.asarray(q, float), np.asarray(p, float)
    keep = q > 0
    with np.errstate(divide="ignore"):
        return float(np.sum(q[keep] * (np.log(q[keep]) - np.log(p[keep]))))


p_true = np.array([0.45, 0.30, 0.20, 0.05])       # T, B, monocytes, NK: composition of a SIMULATED blood sample
q_flat = np.full(4, 0.25)                         # a lazy guess: all four types equally common
q_miss = np.array([0.50, 0.30, 0.20, 0.00])       # a guess that forgets that NK cells exist
print(f"KL(p_true ‖ p_true) = {kl_discrete(p_true, p_true):.3f}   (identical distributions)")
print(f"KL(q_flat ‖ p_true) = {kl_discrete(q_flat, p_true):.3f}")
print(f"KL(p_true ‖ q_flat) = {kl_discrete(p_true, q_flat):.3f}   ← a different number: KL is asymmetric")
print(f"KL(q_miss ‖ p_true) = {kl_discrete(q_miss, p_true):.3f}   ← finite: q may ignore a rare type")
print(f"KL(p_true ‖ q_miss) = {kl_discrete(p_true, q_miss)}     ← infinite: q gives zero probability to something real")

🔍 **What just happened?** The divergence of a distribution from itself is 0, and swapping the arguments changes the
value (0.266 vs 0.194 nats). The last two lines show the asymmetry at its most extreme: a guess that puts **zero**
probability on NK cells has a finite $\mathrm{KL}(q\,\|\,p)$ — averaging under $q$ never visits NK cells, so it never
notices — but an *infinite* $\mathrm{KL}(p\,\|\,q)$, because under the truth NK cells do occur and $q$ calls them
impossible. Which direction we minimise therefore decides which mistakes we tolerate. Variational inference minimises
$\mathrm{KL}(q\,\|\,p)$ (Section 2 shows why), so it tolerates ignoring regions where the true posterior has mass.

Let us see what that means for a continuous posterior with two peaks — say the posterior over a cell's latent "activation"
coordinate when its profile is ambiguous between a resting and an activated state — approximated by a single Gaussian.

In [ ]:
# ── 1.2 Fitting one Gaussian q to a two-peaked p: KL(q‖p) versus KL(p‖q) on a grid ──
z_grid = np.linspace(-8, 8, 1601)
dz = z_grid[1] - z_grid[0]
p_bimodal = 0.5 * stats.norm.pdf(z_grid, -2, 0.6) + 0.5 * stats.norm.pdf(z_grid, 2, 0.6)


def kl_on_grid(q, p):
    """KL(q‖p) for densities tabulated on z_grid (rows of q may be several densities); a Riemann sum."""
    q, p = np.maximum(q, 1e-300), np.maximum(p, 1e-300)          # floor tiny values to avoid log(0)
    return np.sum(q * (np.log(q) - np.log(p)), axis=-1) * dz


mus = np.linspace(-4, 4, 81)                                       # candidate means of q
sds = np.linspace(0.2, 4, 77)                                      # candidate standard deviations of q
kl_qp = np.array([kl_on_grid(stats.norm.pdf(z_grid, m, sds[:, None]), p_bimodal) for m in mus])
kl_pq = np.array([kl_on_grid(p_bimodal, stats.norm.pdf(z_grid, m, sds[:, None])) for m in mus])
i, j = np.unravel_index(np.argmin(kl_qp), kl_qp.shape)
best_qp = (mus[i], sds[j])
i, j = np.unravel_index(np.argmin(kl_pq), kl_pq.shape)
best_pq = (mus[i], sds[j])
print(f"minimising KL(q‖p): q = N({best_qp[0]:+.1f}, {best_qp[1]:.2f}²)   KL = {kl_qp.min():.3f} nats")
print(f"minimising KL(p‖q): q = N({best_pq[0]:+.1f}, {best_pq[1]:.2f}²)   KL = {kl_pq.min():.3f} nats")
print(f"moment matching predicts for KL(p‖q): mean 0, sd = sqrt(0.6² + 2²) = {np.sqrt(0.6**2 + 2**2):.2f}")

In [ ]:
# ── 1.3 Picture: mode-seeking versus mass-covering ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.fill_between(z_grid, p_bimodal, color="0.85", label="p: two cell states (target)")
a1.plot(z_grid, stats.norm.pdf(z_grid, *best_qp), color="tab:blue", lw=2, label="best q under KL(q‖p): one mode")
a1.plot(z_grid, stats.norm.pdf(z_grid, *best_pq), color="tab:orange", lw=2, ls="--",
        label="best q under KL(p‖q): covers both")
a1.set_xlim(-6, 6); a1.set_xlabel("latent coordinate z"); a1.set_ylabel("density")
a1.set_title("Which Gaussian is 'closest' depends on the KL direction"); a1.legend(fontsize=7, loc="upper left")
a2.plot(mus, kl_qp.min(axis=1), color="tab:blue", lw=2, label="KL(q‖p), best σ for each μ")
a2.plot(mus, kl_pq.min(axis=1), color="tab:orange", lw=2, ls="--", label="KL(p‖q), best σ for each μ")
a2.set_ylim(0, 2.5); a2.set_xlabel("mean μ of the Gaussian q"); a2.set_ylabel("KL divergence (nats)")
a2.set_title("KL(q‖p) has two minima (one per mode)"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** We searched 81 × 77 Gaussians for the one closest to the two-peaked target. Under
$\mathrm{KL}(q\,\|\,p)$ the winner sits on **one** peak with the peak's own width, $\mathcal N(-2, 0.60^2)$ (the mirror
image on the right peak is equally good — the right panel shows two equal minima). Under $\mathrm{KL}(p\,\|\,q)$ the winner
is a broad Gaussian centred between the peaks, $\mathcal N(0, 2.10^2)$, whose standard deviation matches the target's
(the grid's nearest value to the moment-matching prediction 2.09).

- $\mathrm{KL}(q\,\|\,p)$ is **mode-seeking** ("zero-forcing"): $q$ is punished wherever it puts mass that $p$ lacks, so it
  hides inside one high-probability region and *underestimates the uncertainty*.
- $\mathrm{KL}(p\,\|\,q)$ is **mass-covering** ("zero-avoiding"): $q$ is punished wherever $p$ has mass it lacks, so it
  spreads over everything, including the valley between the peaks.

⚠️ **Why this matters in biology.** A variational posterior that sits on one mode can confidently report one explanation
of a cell, a phylogeny or a genotype when the data support two. Keep "VI tends to be over-confident" in mind; we will
measure it in Sections 3 and 4.

✏️ **Your turn.** Move the two peaks closer together (e.g. to $\pm1$) and re-run 1.2–1.3. Below roughly which separation
does the $\mathrm{KL}(q\,\|\,p)$ solution switch from one peak to a single Gaussian covering both? (Hint: when the valley
disappears, the target itself becomes unimodal.)

---
## 2 · The evidence lower bound (ELBO)

> 📍 **Where we are.** We want the $q$ that minimises $\mathrm{KL}(q\,\|\,p(z\mid x))$. But that KL contains the
> posterior — the very thing we cannot compute. This section shows how to minimise it anyway, and that you already did
> so in [Module 13](13_mixture_models_em.ipynb) without knowing it.

**The derivation.** Take any distribution $q(z)$. Because $\log p(x)$ does not depend on $z$, it equals its own
average under $q$; then multiply and divide by $q(z)$ inside the log, using $p(x)=p(x,z)/p(z\mid x)$:

$$\log p(x)=\mathbb E_q\!\left[\log\frac{p(x,z)}{p(z\mid x)}\right]
=\underbrace{\mathbb E_q\!\left[\log\frac{p(x,z)}{q(z)}\right]}_{\mathrm{ELBO}(q)}
+\underbrace{\mathbb E_q\!\left[\log\frac{q(z)}{p(z\mid x)}\right]}_{\mathrm{KL}(q\,\|\,p(z\mid x))\;\ge\;0}.$$

*In words:* the log-evidence splits into two pieces for **every** $q$. The second is the KL divergence from $q$ to the
true posterior. Because it is never negative, the first piece is a lower bound on $\log p(x)$ — the **evidence lower
bound**, ELBO. Since $\log p(x)$ is a fixed number, *raising the ELBO is the same as lowering the KL to the posterior* —
and the ELBO contains only $p(x,z)=p(x\mid z)p(z)$, which we *can* evaluate.

Two equivalent ways of writing the ELBO are used constantly:

$$\mathrm{ELBO}(q)=\underbrace{\mathbb E_q[\log p(x,z)]}_{\text{fit}}\;\underbrace{-\;\mathbb E_q[\log q(z)]}_{\text{entropy of }q}
\;=\;\underbrace{\mathbb E_q[\log p(x\mid z)]}_{\text{reconstruction}}\;-\;\underbrace{\mathrm{KL}\big(q(z)\,\|\,p(z)\big)}_{\text{stay close to the prior}}.$$

*In words:* the first form says "put $q$ where the joint probability is high, but stay spread out" (the entropy term
stops $q$ collapsing onto a single point). The second form — used by VAEs — says "explain the data well, but do not
stray further from the prior than the data justify".

> 🔗 **Builds on [Module 13 §5.3](13_mixture_models_em.ipynb).** The lower bound $B(\theta;q)$ you met there, built
> from Jensen's inequality, *is* the ELBO. The E-step set $q$ to the exact posterior (the responsibilities
> $\gamma_{ni}$), which makes the KL zero so the bound touches $\log p(x)$; the M-step raised the bound over $\theta$.
> **EM is coordinate ascent on the ELBO with an exact E-step.** Variational inference keeps the same objective when the
> exact E-step is impossible and $q$ must come from a restricted family.

Let us verify the identity numerically for one cell of the two-state mixture of Module 13.

In [ ]:
# ── 2.1 log p(x) = ELBO(q) + KL(q ‖ p(z|x)) for one cell of a two-state ("off"/"on") mixture ──
pi_mix, mu_mix, sd_mix = np.array([0.65, 0.35]), np.array([2.0, 4.0]), 0.7   # mixture of Module 13
x_cell = 2.9                                                                 # this cell's log-expression
log_joint = np.log(pi_mix) + stats.norm.logpdf(x_cell, mu_mix, sd_mix)     # log p(x, z) for z = off, on
log_px = np.logaddexp(log_joint[0], log_joint[1])                          # log p(x) = log Σ_z p(x, z)
post_on = np.exp(log_joint[1] - log_px)                                    # exact posterior p(z = on | x)


def elbo_two_state(r):
    """ELBO(q) = Σ_z q(z)[log p(x, z) − log q(z)] for q(z = on) = r (vectorised over r)."""
    q = np.stack([1 - r, r])
    return np.sum(q * (log_joint[:, None] - np.log(q)), axis=0)


def kl_to_posterior(r):
    """KL(q ‖ p(z|x)) computed directly from the exact posterior."""
    return (1 - r) * np.log((1 - r) / (1 - post_on)) + r * np.log(r / post_on)


print(f"log p(x) = {log_px:.4f};  exact posterior p(on | x) = {post_on:.3f}\n")
print("   q(on) |   ELBO   |  KL(q‖post) | ELBO + KL")
for r in [0.05, 0.5, post_on, 0.95]:
    e, k = elbo_two_state(np.array([r]))[0], kl_to_posterior(r)
    print(f"   {r:5.3f} | {e:8.4f} |  {k:9.4f}  | {e + k:8.4f}")
r_grid = np.linspace(0.001, 0.999, 999)
print(f"\nq(on) that maximises the ELBO on a grid: {r_grid[np.argmax(elbo_two_state(r_grid))]:.3f}")

🔍 **What just happened?** For every choice of $q$ the ELBO and the KL add up to the same number, $\log p(x)$. The ELBO
is highest — and equal to $\log p(x)$, with KL = 0 — exactly at $q(\text{on})$ = the posterior responsibility (0.264 for
this cell, whose value 2.9 lies between the two means but closer to "off", which is also the more common state). Maximising the ELBO over $q$ found the posterior without our ever
dividing by $p(x)$. With a richer model we cannot reach KL = 0, but the same logic finds the best $q$ in the family.

Now the "EM = coordinate ascent" claim, on 300 SIMULATED cells: alternately maximise the ELBO over $q$ (E-step) and over
the parameters $\theta=(\pi,\mu,\sigma)$ (M-step), and record the ELBO after every half-step.

In [ ]:
# ── 2.2 EM is coordinate ascent on the ELBO: the ELBO rises after every E-step AND every M-step ──
rng = np.random.default_rng(SEED + 2)
n_mix = 300
is_on = rng.random(n_mix) < 0.35
x_mix = np.where(is_on, rng.normal(4.0, 0.7, n_mix), rng.normal(2.0, 0.7, n_mix))   # SIMULATED log-expression


def elbo_mixture(x, resp, pi, mu, sd):
    """ELBO = Σ_n Σ_i γ_ni [log π_i + log N(x_n | μ_i, σ_i²) − log γ_ni] for a 1-D Gaussian mixture."""
    lj = np.log(pi) + stats.norm.logpdf(x[:, None], mu, sd)
    return float(np.sum(resp * (lj - np.log(np.clip(resp, 1e-300, 1)))))


pi_t, mu_t, sd_t = np.array([0.5, 0.5]), np.array([1.0, 5.0]), np.array([1.5, 1.5])   # a rough starting guess
resp = np.full((n_mix, 2), 0.5)
trace = [elbo_mixture(x_mix, resp, pi_t, mu_t, sd_t)]
for it in range(25):
    lj = np.log(pi_t) + stats.norm.logpdf(x_mix[:, None], mu_t, sd_t)                 # E-step: exact posterior
    resp = np.exp(lj - np.logaddexp(lj[:, 0], lj[:, 1])[:, None])
    trace.append(elbo_mixture(x_mix, resp, pi_t, mu_t, sd_t))
    loglik = np.sum(np.logaddexp(lj[:, 0], lj[:, 1]))                                # log p(x | θ) at this θ
    nk = resp.sum(axis=0)                                                              # M-step (Module 13)
    pi_t, mu_t = nk / n_mix, resp.T @ x_mix / nk
    sd_t = np.sqrt(np.sum(resp * (x_mix[:, None] - mu_t) ** 2, axis=0) / nk)
    trace.append(elbo_mixture(x_mix, resp, pi_t, mu_t, sd_t))
    if it < 2:
        print(f"iteration {it + 1}: ELBO after E-step = {trace[-2]:.3f} = log-likelihood {loglik:.3f}")
print(f"ELBO never decreased: {bool(np.all(np.diff(trace) >= -1e-9))}; final μ = {np.round(mu_t, 2)}, "
      f"π = {np.round(pi_t, 2)} (simulated: {100 * is_on.mean():.0f} % of cells 'on')")
gap = trace[-1] - np.array(trace[:-1])                                               # distance to the final ELBO
half = np.arange(len(gap)) / 2
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.semilogy(half, gap, color="0.6", lw=1)
ax.semilogy(half[1::2], gap[1::2], "o", ms=4, color="tab:blue", label="after an E-step")
ax.semilogy(half[2::2], gap[2::2], "s", ms=4, color="tab:orange", label="after an M-step")
ax.set_xlabel("EM iteration"); ax.set_ylabel("final ELBO − current ELBO (nats, log scale)"); ax.legend(fontsize=7)
ax.set_title("Every E-step and every M-step brings the ELBO closer to its maximum (SIMULATED cells)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** After every E-step the ELBO equals the log-likelihood $\log p(x\mid\theta)$ (the two printed
numbers agree), because $q$ is the exact posterior and the KL term vanishes. Each M-step then raises the ELBO further by
moving $\theta$. The plot shows the distance to the final ELBO on a log scale: it shrinks after **every** half-step, E
or M, and never grows. The final estimates are close to the simulated "off" and "on" means (2 and 4) and to the fraction
of "on" cells actually drawn in this sample (39 %, against an expected 35 %). EM is VI with a perfect family; the rest of
this module is about what to do when the family must be imperfect.

✏️ **Your turn.** Replace the exact E-step by a *restricted* one: force every cell's $q$ to be the same, $q(\text{on})=0.35$
for all cells (a terrible but legal family). Is the ELBO still a lower bound on the log-likelihood? How large is the gap
(the summed KL) at the end?

---
## 3 · Mean-field variational inference by hand (CAVI)

> 📍 **Where we are.** The ELBO lets us search for the best $q$ in a family. The simplest family is **mean-field**: all
> hidden variables independent under $q$. Here we derive the classic coordinate-ascent algorithm (CAVI) for a model whose
> exact posterior we also know ([Module 21](21_bayesian_learning_mcmc.ipynb), conjugate priors), so we can see exactly
> what mean-field gets right and wrong.

**The model.** The log-expression $x_1,\dots,x_N$ of one gene in $N=20$ cells of the same type is modelled as Gaussian
with unknown mean $\mu$ and unknown **precision** $\tau=1/\sigma^2$. With the conjugate Normal–Gamma prior

$$x_n\mid\mu,\tau\sim\mathcal N(\mu,\tau^{-1}),\qquad \mu\mid\tau\sim\mathcal N\big(\mu_0,(\lambda_0\tau)^{-1}\big),\qquad
\tau\sim\mathrm{Gamma}(a_0,b_0),$$

*In words:* the cells scatter around the gene's mean $\mu$ with spread $1/\sqrt\tau$; before seeing data we believe $\mu$
is near $\mu_0$ (with an uncertainty that scales with the noise level) and that $\tau$ is around $a_0/b_0$. The exact
posterior is again Normal–Gamma (a standard conjugate result), so $\mu$ and $\tau$ are **dependent** a posteriori: when
the noise is large ($\tau$ small), $\mu$ is less certain.

**The mean-field family and the CAVI rule.** We approximate $p(\mu,\tau\mid x)\approx q(\mu)\,q(\tau)$. Maximising the ELBO
over one factor while holding the other fixed has a general solution (Bishop 2006, §10.1.1):

$$\log q^*(\mu)=\mathbb E_{q(\tau)}\big[\log p(x,\mu,\tau)\big]+\text{const},\qquad
\log q^*(\tau)=\mathbb E_{q(\mu)}\big[\log p(x,\mu,\tau)\big]+\text{const}.$$

*In words:* the optimal factor for one variable is the log-joint with the *other* variables averaged out under their
current $q$.

> 🔗 **Builds on [Module 21 §6](21_bayesian_learning_mcmc.ipynb).** Gibbs sampling updates one variable at a time by
> *drawing* it from its full conditional given the current *values* of the others. CAVI updates one factor at a time by
> setting it to the exponentiated *expected* log full conditional given the current *distributions* of the others. Same
> coordinate-wise structure: Gibbs samples, CAVI averages — and CAVI converges to a fixed answer instead of a cloud of samples.

For our model the averages are easy and give named distributions:

$$q(\mu)=\mathcal N\!\Big(\mu_N,\;\big[(\lambda_0+N)\,\mathbb E[\tau]\big]^{-1}\Big),\quad \mu_N=\frac{\lambda_0\mu_0+N\bar x}{\lambda_0+N};
\qquad q(\tau)=\mathrm{Gamma}\big(a_q,b_q\big),\quad a_q=a_0+\tfrac{N+1}{2},\;\;
b_q=b_0+\tfrac12\,\mathbb E_{q(\mu)}\Big[\textstyle\sum_n(x_n-\mu)^2+\lambda_0(\mu-\mu_0)^2\Big].$$

*In words:* $q(\mu)$ is the familiar precision-weighted average of prior mean and data mean, with a precision that uses
the *current average* noise precision $\mathbb E[\tau]=a_q/b_q$; $q(\tau)$ counts squared deviations from $\mu$, averaged
over $q(\mu)$ (using $\mathbb E[(x_n-\mu)^2]=(x_n-\mu_N)^2+\mathrm{Var}_q(\mu)$). Each update needs the other, so we iterate.

In [ ]:
# ── 3.1 SIMULATED data, the exact Normal–Gamma posterior and CAVI ──
rng = np.random.default_rng(SEED + 3)
x_gene = rng.normal(2.0, 0.6, 20)            # one gene in 20 cells: true μ = 2.0, true τ = 1/0.6² ≈ 2.78
N_g, xbar = len(x_gene), x_gene.mean()
mu0, lam0, a0, b0 = 1.0, 0.1, 1.0, 0.2       # weak prior: μ near 1 (λ0 = 0.1 'pseudo-cells'), τ broad

lam_N = lam0 + N_g                            # exact posterior: Normal–Gamma(μ_N, λ_N, a_N, b_N)
mu_N = (lam0 * mu0 + N_g * xbar) / lam_N
a_N = a0 + N_g / 2
b_N = b0 + 0.5 * np.sum((x_gene - xbar) ** 2) + lam0 * N_g * (xbar - mu0) ** 2 / (2 * lam_N)


def cavi_normal_gamma(x, n_sweeps=8, e_tau=1.0):
    """Coordinate ascent for q(μ) q(τ); returns (m, λ_q, a_q, b_q) after every sweep."""
    n, s1, s2 = len(x), x.sum(), np.sum(x ** 2)
    history = []
    for _ in range(n_sweeps):
        m = (lam0 * mu0 + s1) / (lam0 + n)                   # update q(μ) = N(m, 1/λ_q)
        lam_q = (lam0 + n) * e_tau
        e_mu, e_mu2 = m, m ** 2 + 1 / lam_q                  # moments of q(μ) needed by q(τ)
        a_q = a0 + (n + 1) / 2                                # update q(τ) = Gamma(a_q, b_q)
        b_q = b0 + 0.5 * (s2 - 2 * e_mu * s1 + n * e_mu2 + lam0 * (e_mu2 - 2 * mu0 * e_mu + mu0 ** 2))
        e_tau = a_q / b_q
        history.append((m, lam_q, a_q, b_q))
    return history


mu_ax, tau_ax = np.linspace(mu_N - 1.0, mu_N + 1.0, 201), np.linspace(0.02, 5.0, 250)
MU, TAU = np.meshgrid(mu_ax, tau_ax)
cell_area = (mu_ax[1] - mu_ax[0]) * (tau_ax[1] - tau_ax[0])
log_exact = stats.gamma.logpdf(TAU, a_N, scale=1 / b_N) + stats.norm.logpdf(MU, mu_N, 1 / np.sqrt(lam_N * TAU))
print("sweep |  mean of q(μ)  sd of q(μ) |  E_q[τ]  | KL(q ‖ exact posterior)")
for k, (m, lam_q, a_q, b_q) in enumerate(cavi_normal_gamma(x_gene, n_sweeps=6), 1):
    log_q = stats.norm.logpdf(MU, m, 1 / np.sqrt(lam_q)) + stats.gamma.logpdf(TAU, a_q, scale=1 / b_q)
    kl = np.sum(np.exp(log_q) * (log_q - log_exact)) * cell_area
    print(f"  {k}   |     {m:.4f}      {1 / np.sqrt(lam_q):.4f}  |  {a_q / b_q:.4f} |  {kl:.5f}")

🔍 **What just happened?** The mean of $q(\mu)$ is exact from the first sweep (it does not depend on $\tau$ in this
model); the spread of $q(\mu)$ and the expected precision settle within three sweeps, and the KL to the exact posterior —
evaluated on a grid, something we can only do because the exact answer is known — drops from 0.15 after the first sweep
to a floor of about 0.023 nats that no mean-field $q$ can go below. That floor is the price of assuming independence. Let us look at
where the remaining discrepancy lives.

In [ ]:
# ── 3.2 🧪 Ground-truth check: exact posterior versus the mean-field approximation ──
m, lam_q, a_q, b_q = cavi_normal_gamma(x_gene, n_sweeps=20)[-1]
q_joint = stats.norm.pdf(MU, m, 1 / np.sqrt(lam_q)) * stats.gamma.pdf(TAU, a_q, scale=1 / b_q)
sd_mu_exact = np.sqrt(b_N / (lam_N * (a_N - 1)))           # marginal of μ is a Student-t
sd_mu_vi = 1 / np.sqrt(lam_q)
print(f"posterior mean of μ : exact {mu_N:.3f} | mean-field {m:.3f}   (truth used to simulate: 2.0)")
print(f"posterior mean of τ : exact {a_N / b_N:.3f} | mean-field {a_q / b_q:.3f}   (truth: {1 / 0.6**2:.2f}; "
      f"these 20 cells have sample sd {x_gene.std(ddof=1):.2f} → 1/sd² = {1 / x_gene.var(ddof=1):.2f})")
print(f"posterior sd of μ   : exact {sd_mu_exact:.4f} | mean-field {sd_mu_vi:.4f}  → underestimated by "
      f"{100 * (1 - sd_mu_vi / sd_mu_exact):.1f} %")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
levels = np.exp(log_exact).max() * np.array([0.05, 0.25, 0.5, 0.75, 0.95])   # same contour heights for both
a1.contourf(MU, TAU, np.exp(log_exact), levels=np.r_[levels, 1.01 * np.exp(log_exact).max()], cmap="Greys")
a1.contour(MU, TAU, q_joint, levels=levels, colors="tab:blue", linewidths=1)
a1.set_xlabel("mean μ"); a1.set_ylabel("precision τ"); a1.set_ylim(0.2, 3.8)
a1.set_title("exact posterior (grey) vs mean-field q(μ)q(τ) (blue)")
marg_exact = stats.t.pdf(mu_ax, df=2 * a_N, loc=mu_N, scale=np.sqrt(b_N / (a_N * lam_N)))
a2.plot(mu_ax, marg_exact, color="0.3", lw=2, label="exact marginal p(μ | x) (Student-t)")
a2.plot(mu_ax, stats.norm.pdf(mu_ax, m, sd_mu_vi), color="tab:blue", ls="--", lw=2, label="mean-field q(μ)")
a2.axvline(2.0, color="tab:red", lw=1, label="true μ (simulation)")
a2.set_xlabel("mean μ"); a2.set_ylabel("density"); a2.legend(fontsize=7)
a2.set_title("marginal of μ: right centre, slightly too narrow")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The exact and mean-field posterior means of $\mu$ and of $\tau$ agree to three decimals. Both
differ from the values used in the simulation ($\mu=2.0$, $\tau=2.78$) because of the data, not the approximation: 20
cells are a small sample, and these happen to scatter more than average (sample sd 0.76 instead of 0.6). The **shape** is
where mean-field fails. The exact posterior (grey) is pear-shaped: at low
precision $\tau$ (bottom) the plausible values of $\mu$ spread wider, at high precision (top) they are packed more
tightly, because the noise level sets how well 20 cells pin down the mean. In the mean-field product (blue) the spread of
$\mu$ is by construction the same at every $\tau$ — its contours are ovals stretched only by the skew of $q(\tau)$ — so it
cannot represent this dependence, and its marginal for $\mu$ is a Gaussian about 5 % narrower (4.7 %) than the exact Student-t.

Five percent is small because with 20 cells this posterior is *almost* a product. The general lesson, which Section 4
makes dramatic: **mean-field VI recovers posterior means well but underestimates posterior variances, and misses
correlations completely.** The stronger the dependence between hidden variables, the worse the underestimate.

⚠️ **CAVI needs conjugacy.** Every update above was a named distribution because the model is conditionally conjugate.
Replace the Gaussian likelihood by a negative binomial, or the mean by a neural network, and the expectations no longer
have closed forms. The next section removes this restriction.

✏️ **Your turn.** Re-run 3.1–3.2 with only 5 cells (`x_gene = rng.normal(2.0, 0.6, 5)`). How much does the mean-field
standard deviation of $\mu$ now underestimate the exact one, and why does the pear shape get more pronounced?

---
## 4 · Stochastic variational inference and the reparameterisation trick

> 📍 **Where we are.** CAVI works only for conjugate models. The modern recipe (*black-box* or *stochastic* VI) works for
> almost any model: choose a parametric family $q_\phi(z)$ — e.g. Gaussians with means and standard deviations $\phi$ —
> and climb the ELBO by **gradient ascent** ([Module 04](04_calculus_and_optimization.ipynb)), estimating the gradient with
> random samples. The difficulty: the ELBO is an *expectation under $q_\phi$*, and the distribution itself depends on $\phi$.

We need $\nabla_\phi\,\mathbb E_{q_\phi(z)}[f(z)]$ for a function $f$ (for the ELBO, $f(z)=\log p(x,z)-\log q_\phi(z)$).
There are two classic Monte Carlo estimators.

**1 · The score-function (REINFORCE) estimator.** Move the gradient inside the integral and use
$\nabla_\phi q_\phi=q_\phi\nabla_\phi\log q_\phi$:

$$\nabla_\phi\,\mathbb E_{q_\phi}[f(z)]=\int f(z)\,\nabla_\phi q_\phi(z)\,dz=\mathbb E_{q_\phi}\big[f(z)\,\nabla_\phi\log q_\phi(z)\big]
\approx\frac1S\sum_{s=1}^S f(z_s)\,\nabla_\phi\log q_\phi(z_s),\quad z_s\sim q_\phi .$$

*In words:* sample from $q$, and weight each sample's $f$-value by how much the parameters would change its probability.
It needs no derivative of $f$ — but it multiplies two noisy quantities and has high variance.

**2 · The reparameterisation estimator.** Write the random variable as a deterministic function of parameter-free noise:
for $q_\phi=\mathcal N(\mu,\sigma^2)$, $z=\mu+\sigma\varepsilon$ with $\varepsilon\sim\mathcal N(0,1)$. Then the expectation is
over $\varepsilon$, which does not depend on $\phi$, and the gradient goes inside:

$$\nabla_\mu\,\mathbb E_{q}[f(z)]=\mathbb E_\varepsilon\big[f'(\mu+\sigma\varepsilon)\big],\qquad
\nabla_\sigma\,\mathbb E_{q}[f(z)]=\mathbb E_\varepsilon\big[f'(\mu+\sigma\varepsilon)\,\varepsilon\big].$$

*In words:* push the noise through the model and differentiate the result with the chain rule. It uses the *slope* of
$f$, which carries far more information per sample than its value alone.

> 🔗 **Builds on [Module 22 §7](22_neural_networks.ipynb).** There you checked that PyTorch's autograd reproduces
> hand-derived backpropagation. With $z=\mu+\sigma\varepsilon$ written in code, autograd computes the reparameterised
> gradient for free: the random draw becomes just another node in the computation graph, with $\mu$ and $\sigma$ as
> ordinary parameters upstream of it.

🧮 **Math refresher — why $z=\mu+\sigma\varepsilon$ has the right distribution.** Adding $\mu$ shifts a distribution and
multiplying by $\sigma>0$ stretches it: if $\varepsilon$ has mean 0 and variance 1, then $\mu+\sigma\varepsilon$ has mean
$\mu$ and variance $\sigma^2$, and a shifted, stretched Gaussian is still Gaussian. Sampling $z$ this way is exactly as
valid as sampling from $\mathcal N(\mu,\sigma^2)$ directly.

**A biological test case.** A gene has $x=12$ UMIs in a cell; its log-rate $z$ has variational distribution
$q=\mathcal N(1.5,0.5^2)$, and $f(z)=\log\mathrm{Poisson}(12\mid e^{z})$. The true gradient with respect to $\mu$ is known in
closed form, $x-\mathbb E_q[e^z]=x-e^{\mu+\sigma^2/2}$, so we can measure each estimator's error.

In [ ]:
# ── 4.1 Two Monte Carlo estimators of d/dμ E_q[log Poisson(x | e^z)]: same mean, very different noise ──
rng = np.random.default_rng(SEED + 4)
x_count, mu_q, sd_q = 12, 1.5, 0.5


def log_poisson(z):
    """log Poisson(x_count | rate e^z): the log-likelihood of the observed count for log-rate z."""
    return x_count * z - np.exp(z) - gammaln(x_count + 1)


true_grad = x_count - np.exp(mu_q + sd_q ** 2 / 2)          # exact gradient
eps = rng.standard_normal(100_000)
z_s = mu_q + sd_q * eps                                     # reparameterised samples z = μ + σ ε
score_est = log_poisson(z_s) * (z_s - mu_q) / sd_q ** 2     # f(z) · d log q(z) / dμ
reparam_est = x_count - np.exp(z_s)                         # f'(z) = d f(μ + σ ε) / dμ
print(f"true gradient: {true_grad:.3f}")
print(f"score-function  : mean {score_est.mean():.3f}, sd of a single-sample estimate {score_est.std():.2f}")
print(f"reparameterised : mean {reparam_est.mean():.3f}, sd of a single-sample estimate {reparam_est.std():.2f}")
print(f"variance ratio (score / reparameterised): {score_est.var() / reparam_est.var():.0f}×")
for S in (1, 10, 100):
    print(f"  averaging S = {S:3d} samples → sd {score_est.std() / np.sqrt(S):6.2f} (score) vs "
          f"{reparam_est.std() / np.sqrt(S):5.2f} (reparameterised)")
fig, ax = plt.subplots(figsize=(6.4, 3.2))
bins = np.linspace(-60, 60, 121)
ax.hist(score_est, bins=bins, density=True, alpha=0.6, color="tab:orange", label="score-function")
ax.hist(reparam_est, bins=bins, density=True, alpha=0.6, color="tab:blue", label="reparameterised")
ax.axvline(true_grad, color="k", lw=1.5, label="true gradient")
ax.set_xlabel("single-sample gradient estimate"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("Both are unbiased; the reparameterised one is far less noisy")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both estimators average to the true gradient (6.92): both are **unbiased**. But one sample
of the score-function estimator has a standard deviation of about 19, against about 2.7 for the reparameterised one — a
variance about 50 times larger. To reach the same precision the score-function estimator needs roughly 50 times more
samples. This is why the reparameterisation trick (Kingma & Welling 2014; Rezende *et al.* 2014) made deep generative
models trainable: one random sample per cell per step is enough.

⚠️ **When the trick is unavailable.** Discrete latent variables (a cluster label, a genotype) cannot be written as a
smooth function of parameter-free noise. Then one falls back on score-function estimators with variance-reduction tricks,
continuous relaxations, or sums over the discrete values — which is what EM's exact E-step does.

Now the full recipe on a problem with a known answer: **Bayesian logistic regression** for a case–control study
([Module 09](09_logistic_regression.ipynb) + [Module 21](21_bayesian_learning_mcmc.ipynb)). The posterior over the SNP
effects has no closed form, so we compare stochastic VI with a long Metropolis–Hastings run, which is slow but
asymptotically exact. To make the comparison informative, one null SNP is in strong **linkage disequilibrium** (LD) with
a causal one — like the two linked SNPs of [Module 21 §3.3](21_bayesian_learning_mcmc.ipynb).

In [ ]:
# ── 4.2 SIMULATED case–control data: 5 SNPs, one null SNP in strong LD with a causal SNP ──
rng = np.random.default_rng(SEED + 7)
n_people = 400
G_snps, _, _ = simulate_genotypes(rng, n_people, 4)
G_snps = G_snps.astype(float)
ld_partner = G_snps[:, 0].copy()                          # inherited together with SNP1 ...
recomb = rng.random(n_people) < 0.1                       # ... except on ~10 % of chromosomes
ld_partner[recomb] = rng.binomial(2, G_snps[:, 0].mean() / 2, recomb.sum())
X_snp = np.column_stack([np.ones(n_people), G_snps[:, 0], ld_partner, G_snps[:, 1:]])
coef_names = ["intercept", "SNP1 (causal)", "SNP1-LD (null)", "SNP2 (causal)", "SNP3 (causal)", "SNP4 (null)"]
beta_true = np.array([-1.0, 0.8, 0.0, -0.6, 0.5, 0.0])
y_case = (rng.random(n_people) < sigmoid(X_snp @ beta_true)).astype(float)
print(f"SIMULATED: {n_people} people, {int(y_case.sum())} cases; "
      f"correlation SNP1 vs its LD partner r = {np.corrcoef(X_snp[:, 1], X_snp[:, 2])[0, 1]:.2f}")

In [ ]:
# ── 4.3 Mean-field Gaussian VI for Bayesian logistic regression, in PyTorch ──
PRIOR_SD = 2.0                                                      # prior β_j ~ N(0, 2²)
X_t, y_t = torch.tensor(X_snp, dtype=torch.float32), torch.tensor(y_case, dtype=torch.float32)


def fit_vi_logistic(X, y, n_steps=2000, n_samples=32, lr=0.05, seed=SEED):
    """Maximise the ELBO over q(β) = Π_j N(m_j, s_j²) with reparameterised samples and Adam."""
    torch.manual_seed(seed)
    m = torch.zeros(X.shape[1], requires_grad=True)
    log_s = torch.full((X.shape[1],), -1.0, requires_grad=True)       # optimise log s so that s stays positive
    opt = torch.optim.Adam([m, log_s], lr=lr)
    trace = []
    for _ in range(n_steps):
        beta = m + torch.exp(log_s) * torch.randn(n_samples, X.shape[1])   # S draws β = m + s·ε
        eta = beta @ X.T                                                   # (S × people) linear predictors
        log_lik = (y * eta - nn.functional.softplus(eta)).sum(dim=1)       # Bernoulli log-likelihood
        log_prior = -0.5 * (beta ** 2).sum(dim=1) / PRIOR_SD ** 2          # Gaussian prior (constants dropped)
        elbo = (log_lik + log_prior).mean() + log_s.sum()                  # + entropy of q (constants dropped)
        opt.zero_grad(); (-elbo).backward(); opt.step()
        trace.append(elbo.item())
    return m.detach().numpy(), torch.exp(log_s).detach().numpy(), np.array(trace)


t0 = time.time()
vi_mean, vi_sd, vi_trace = fit_vi_logistic(X_t, y_t)
print(f"VI: 2,000 Adam steps in {time.time() - t0:.1f} s; ELBO estimate {vi_trace[:50].mean():.1f} (first 50 steps) "
      f"→ {vi_trace[-200:].mean():.1f} (last 200)")

In [ ]:
# ── 4.4 The gold standard: random-walk Metropolis–Hastings (Module 21) ──
def log_post_logistic(beta):
    """Unnormalised log posterior: Bernoulli log-likelihood + N(0, PRIOR_SD²) log prior."""
    eta = X_snp @ beta
    return np.sum(y_case * eta - np.logaddexp(0, eta)) - 0.5 * np.sum(beta ** 2) / PRIOR_SD ** 2


def metropolis_logistic(n_iter=40_000, step=0.12, burn_in=5_000, thin=5, seed=SEED + 17):
    """Gaussian random-walk proposals, accepted with probability min(1, p(β')/p(β))."""
    gen = np.random.default_rng(seed)
    beta, lp = np.zeros(X_snp.shape[1]), log_post_logistic(np.zeros(X_snp.shape[1]))
    kept, n_acc = [], 0
    for it in range(n_iter):
        prop = beta + step * gen.standard_normal(beta.size)
        lp_prop = log_post_logistic(prop)
        if np.log(gen.random()) < lp_prop - lp:
            beta, lp, n_acc = prop, lp_prop, n_acc + 1
        if it >= burn_in and it % thin == 0:
            kept.append(beta.copy())
    return np.array(kept), n_acc / n_iter


t0 = time.time()
mh_samples, acc_rate = metropolis_logistic()
print(f"MH: 40,000 iterations in {time.time() - t0:.1f} s, acceptance {acc_rate:.2f}, {len(mh_samples)} samples kept")
print(f"posterior correlation of SNP1 and its LD partner: {np.corrcoef(mh_samples[:, 1], mh_samples[:, 2])[0, 1]:.2f}")

In [ ]:
# ── 4.5 🧪 Ground-truth check: VI versus MCMC versus the true effects ──
mh_lo, mh_hi = np.percentile(mh_samples, [2.5, 97.5], axis=0)
vi_lo, vi_hi = vi_mean - 1.96 * vi_sd, vi_mean + 1.96 * vi_sd
table = pd.DataFrame({"true β": beta_true, "MH mean": mh_samples.mean(0), "VI mean": vi_mean,
                      "MH sd": mh_samples.std(0), "VI sd": vi_sd,
                      "MH 95% CI covers truth": (mh_lo <= beta_true) & (beta_true <= mh_hi),
                      "VI 95% CI covers truth": (vi_lo <= beta_true) & (beta_true <= vi_hi),
                      "MH CI excludes 0": (mh_lo > 0) | (mh_hi < 0),
                      "VI CI excludes 0": (vi_lo > 0) | (vi_hi < 0)}, index=coef_names)
print(table.round(2).to_string())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.scatter(mh_samples[::4, 1], mh_samples[::4, 2], s=3, color="0.6", label="MH samples (≈ exact posterior)")
t = np.linspace(0, 2 * np.pi, 200)
a1.plot(vi_mean[1] + 2 * vi_sd[1] * np.cos(t), vi_mean[2] + 2 * vi_sd[2] * np.sin(t), color="tab:blue", lw=2,
        label="mean-field VI (2 sd ellipse)")
a1.plot(*beta_true[1:3], "r*", ms=12, label="true effects")
a1.set_xlabel("β for SNP1 (causal)"); a1.set_ylabel("β for SNP1-LD partner (null)"); a1.legend(fontsize=7)
a1.set_title("Linked SNPs: a long diagonal ridge VI cannot follow")
ypos = np.arange(len(coef_names))
a2.errorbar(table["MH mean"], ypos + 0.15, xerr=[table["MH mean"] - mh_lo, mh_hi - table["MH mean"]], fmt="o",
            color="0.4", label="MH 95% interval")
a2.errorbar(vi_mean, ypos - 0.15, xerr=1.96 * vi_sd, fmt="o", color="tab:blue", label="VI 95% interval")
a2.plot(beta_true, ypos, "r*", ms=10, label="truth")
a2.axvline(0, color="k", lw=0.5); a2.set_yticks(ypos); a2.set_yticklabels(coef_names, fontsize=8)
a2.set_xlabel("effect size (log-odds per allele)"); a2.legend(fontsize=7, loc="lower right")
a2.set_title("Similar centres; VI intervals far too narrow for linked SNPs")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The posterior **means** from VI and from 40,000 MCMC steps agree closely for every coefficient.
The **uncertainties** do not. For the two linked SNPs the true posterior (grey cloud) is a long diagonal ridge
(correlation −0.93): the data say "SNP1 *or* its partner carries an effect, but cannot tell which", so each individual
effect is very uncertain (MH sd ≈ 0.44). In this sample chance has tilted the ridge towards the *wrong* SNP: both methods
put most of the effect on the null LD partner (means 0.62 and 0.66) and little on the causal SNP1 (0.15 and 0.13). MH's
intervals are honest — wide enough to cover both true values, and excluding zero for neither SNP. Mean-field VI must use
an axis-aligned Gaussian and, minimising $\mathrm{KL}(q\,\|\,p)$, squeezes itself *inside* the ridge: its standard
deviations for these two SNPs (0.11) are about 4 times too small. As a result its 95 % interval declares the **null** SNP
clearly non-zero and **excludes the true effect** of the causal SNP1. A fine-mapping analysis based on these intervals would
confidently name the wrong variant. For the unlinked SNPs (SNP2–SNP4) VI's intervals are somewhat too narrow but still
cover the truth.

This is mode-seeking KL (Section 1) and missing correlations (Section 3) in a case that matters for genetics:
**fine-mapping in LD regions is exactly where mean-field approximations are least trustworthy.**

✏️ **Your turn.** Make $q$ a *full-covariance* Gaussian: parameterise $\beta=m+L\varepsilon$ with a lower-triangular
matrix $L$ (`torch.tril`, with positive diagonal via `exp`), and replace the entropy term by `torch.log(torch.diagonal(L)).sum()`.
Does the VI ellipse now follow the ridge, and do the standard deviations match MH?

### 📄 Research connection — Variational inference as a workhorse of Bayesian machine learning

- **What the field does.** Variational methods entered machine learning in the 1990s for graphical models (Jordan *et al.* 1999), with
    mean-field updates of the kind derived in Section 3 (textbook treatment: Bishop 2006, ch. 10). Stochastic VI (Hoffman
    *et al.* 2013) made VI scale to huge data sets by using mini-batches; black-box VI (Ranganath *et al.* 2014) and the
    reparameterisation trick (Kingma & Welling 2014; Rezende *et al.* 2014) made it work for almost any differentiable
    model. Probabilistic programming languages (Stan's ADVI, Pyro, NumPyro) now run VI automatically from a model
    description. Blei, Kucukelbir & McAuliffe (2017) is the standard review.
- **What this notebook builds.** CAVI for a conjugate model with a known exact posterior, the two gradient estimators with measured variances,
    and reparameterised mean-field VI for logistic regression validated against Metropolis–Hastings. Production systems
    add better optimisers, control variates, richer families (full-rank Gaussians, normalising flows) and diagnostics,
    but the objective — the ELBO — and the estimator are the ones you wrote.
- **Where the frontier is.** Open problems include **assessing** how good a variational approximation is without the exact posterior
    (importance-sampling diagnostics such as Pareto-smoothed importance sampling are one answer), richer yet scalable
    families, and alternative divergences that are less over-confident than $\mathrm{KL}(q\,\|\,p)$. In genetics and
    phylogenetics, VI is used where MCMC is too slow, with the caveat demonstrated in Section 4: uncertainty for strongly
    correlated parameters must be checked.
- **Read next.** Blei, Kucukelbir & McAuliffe (2017) *Variational inference: a review for statisticians*; Kingma & Welling
    (2014) *Auto-encoding variational Bayes*; Bishop (2006) *Pattern Recognition and Machine Learning*, ch. 10.

---
## 5 · Amortised inference: the variational autoencoder

> 📍 **Where we are.** Stochastic VI fits a $q$ for *one* posterior. A single-cell data set has one latent vector $z_c$
> per cell, so per-cell VI would need its own $(\mu_c,\sigma_c)$ for each of millions of cells — and a fresh optimisation
> for every new cell. The VAE solves this with one idea, **amortisation**, and adds a neural-network likelihood. Then we
> make the likelihood a **negative binomial for raw counts**, which gives a miniature scVI.

**Amortisation.** Instead of storing $(\mu_c,\sigma_c)$ for every cell, train a neural network — the **encoder** — that
maps a cell's profile to the parameters of its approximate posterior:

$$q_\phi(z\mid x)=\mathcal N\big(\mu_\phi(x),\;\mathrm{diag}\,\sigma^2_\phi(x)\big).$$

*In words:* the encoder is a learned *formula* for the E-step. In EM, Bayes' rule gave each cell's responsibilities by a
formula; here a network learns a formula that outputs each cell's approximate posterior in one forward pass — for training
cells and for cells it has never seen. The cost of inference is paid once, during training, and "amortised" over all
future cells.

**The generative model (decoder).** A latent cell state $z\sim\mathcal N(0,I)$ (10 numbers) is mapped by a second network,
the **decoder** $f_\theta$, to the parameters of the distribution of the cell's expression, $p_\theta(x\mid z)$.

> 🔗 **Builds on [Module 14](14_pca_dimensionality_reduction.ipynb) and [Module 22 §10](22_neural_networks.ipynb).** With
> a *linear* decoder $x=Wz+\bar x+\text{noise}$ this is probabilistic PCA (Tipping & Bishop 1999), whose maximum-likelihood
> $W$ spans the principal subspace. The VAE replaces the straight line by a curved map — and, below, the Gaussian noise
> by a count distribution.

**The objective.** For each cell, the ELBO of Section 2 in its "reconstruction − KL" form:

$$\mathrm{ELBO}(x)=\mathbb E_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]-\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,\mathcal N(0,I)\big),
\qquad \mathrm{KL}=\tfrac12\sum_{k}\big(\mu_k^2+\sigma_k^2-1-\log\sigma_k^2\big).$$

*In words:* the first term asks the decoder to make the observed profile likely when fed codes drawn from the encoder
(estimated with **one** reparameterised sample $z=\mu+\sigma\varepsilon$ per cell); the second, available in closed form for
Gaussians (you checked it by Monte Carlo in Module 22), keeps codes near the standard-normal prior. Encoder parameters
$\phi$ and decoder parameters $\theta$ are trained **together** by stochastic gradient ascent on the ELBO summed over
mini-batches of cells.

🔬 **Assay box — droplet single-cell RNA-seq counts (details in [Module 30](30_single_cell_rnaseq.ipynb)).**
*What is measured:* for each cell, the number of distinct mRNA molecules (UMIs, unique molecular identifiers) captured
from each gene. *How:* cells are encapsulated in droplets with barcoded beads; mRNA is reverse-transcribed with a cell
barcode and a UMI, pooled and sequenced; reads are collapsed to UMIs per gene per cell. *Typical sizes:* $10^3$–$10^6$
cells × ~20,000 genes; 1,000–10,000 UMIs per cell, so most entries are 0 or small integers. *Artefacts:* each cell's
**library size** (total UMIs) varies several-fold for technical reasons; counts show **overdispersion** beyond Poisson
noise; **batches** (processing days, chemistry versions, labs) shift expression systematically; empty droplets,
doublets, damaged cells and ambient RNA (Module 30). Our simulation includes library-size variation, negative-binomial
noise, donor effects, interferon stimulation and (in Section 7) batches; it is shallow on purpose (~800 UMIs per cell
over 1,000 genes), which makes noise matter.

In [ ]:
# ── 5.1 SIMULATED PBMC-like UMI counts: 3,000 cells, 6 known cell types, one batch ──
rng = np.random.default_rng(SEED + 7)
sc = simulate_scrna(rng, n_cells=3000, n_genes=1000, n_batches=1, depth=800)
obs = sc["obs"]
lognorm_all = log_normalize(sc["counts"])
hvg = np.argsort(lognorm_all.var(axis=0))[::-1][:500]         # 500 most variable genes (Module 30 refines this)
counts = sc["counts"][:, hvg].astype(np.float32)               # raw counts: what the NB decoder explains
lognorm = lognorm_all[:, hvg].astype(np.float32)               # log-normalised: encoder input, Gaussian VAE target
lib_size = counts.sum(axis=1, keepdims=True)                   # observed library size ℓ (UMIs in these genes)
cell_type = obs["true_type"].to_numpy(str)
condition = obs["condition"].to_numpy(str)
print(f"SIMULATED counts: {counts.shape[0]} cells × {counts.shape[1]} genes | {100 * (counts == 0).mean():.0f} % zeros")
print(f"library size ℓ: median {np.median(lib_size):.0f}, 5th–95th percentile "
      f"{np.percentile(lib_size, 5):.0f}–{np.percentile(lib_size, 95):.0f} UMIs")
print("true cell types:", pd.Series(cell_type).value_counts().sort_index().to_dict())

🧮 **Math refresher — the negative binomial (recap of [Module 24](24_glm_count_data.ipynb)).** A count $x$ with mean $\mu$
and *inverse dispersion* $\theta>0$ has

$$\log\mathrm{NB}(x\mid\mu,\theta)=\log\Gamma(x+\theta)-\log\Gamma(\theta)-\log x!+\theta\log\frac{\theta}{\theta+\mu}+x\log\frac{\mu}{\theta+\mu},
\qquad\mathrm{Var}(x)=\mu+\frac{\mu^2}{\theta}.$$

*In words:* a Poisson count whose rate itself varies from cell to cell (a Gamma–Poisson mixture); $\Gamma$ is the gamma
function ($\Gamma(n)=(n-1)!$ for whole numbers), computed on the log scale by `torch.lgamma`. Small $\theta$ means strong
overdispersion; $\theta\to\infty$ recovers the Poisson.

**The mini-scVI decoder.** For cell $c$ and gene $g$, the decoder outputs *proportions* $\rho_{cg}(z)$ that sum to 1 over
genes (a softmax), and the NB mean is the observed library size times that proportion:

$$\mu_{cg}=\ell_c\,\rho_{cg}(z_c),\qquad x_{cg}\sim\mathrm{NB}(\mu_{cg},\theta_g).$$

*In words:* the latent state decides *what fraction of the cell's transcripts* each gene receives; how many transcripts
were captured in total is a technical fact we observe and simply multiply in. The gene-specific $\theta_g$ are learned
parameters. This is how scVI separates biology ($z$) from sequencing depth ($\ell$) — the counts are modelled *raw*, with
no pseudo-count, log or normalisation step.

In [ ]:
# ── 5.2 The negative-binomial log-likelihood in PyTorch, checked against SciPy ──
def nb_log_prob(x, mu, theta, eps=1e-8):
    """log NB(x | mean mu, inverse dispersion theta), elementwise on tensors; Var = mu + mu²/theta."""
    log_theta_mu = torch.log(theta + mu + eps)
    return (torch.lgamma(x + theta) - torch.lgamma(theta) - torch.lgamma(x + 1)
            + theta * (torch.log(theta + eps) - log_theta_mu) + x * (torch.log(mu + eps) - log_theta_mu))


x_test, mu_test = torch.tensor([0.0, 1.0, 5.0, 30.0]), torch.tensor([0.5, 2.0, 5.0, 12.0])
ours = nb_log_prob(x_test, mu_test, torch.tensor(3.0)).numpy()
ref = stats.nbinom.logpmf(x_test.numpy(), n=3.0, p=3.0 / (3.0 + mu_test.numpy()))   # SciPy's (n, p) form
print("ours :", np.round(ours, 4)); print("SciPy:", np.round(ref, 4)); print("agree:", np.allclose(ours, ref, atol=1e-4))

**The architecture.** Encoder: genes → 128 hidden units (ReLU) → two heads, $\mu_\phi(x)$ and $\log\sigma^2_\phi(x)$, each
of length 10. Decoder: 10 → 128 (ReLU) → one output per gene. The *Gaussian VAE* treats the decoder output as the mean of
the log-normalised data, with a learned per-gene variance. The *NB VAE* passes it through a softmax to get $\rho(z)$,
multiplies by $\ell$ and scores the **raw counts** with `nb_log_prob`. Both encoders see the log-normalised profile.
An optional one-hot batch vector is appended to the inputs of encoder and decoder; we use it in Section 7 (and again in
[Module 31](31_batch_integration.ipynb)).

In [ ]:
# ── 5.3 A small VAE with a Gaussian or a negative-binomial decoder (optionally batch-aware) ──
class MiniVAE(nn.Module):
    """Encoder q(z|x) = N(μ(x), diag σ²(x)); decoder p(x|z): Gaussian (log data) or NB (raw counts, mean ℓ·ρ(z))."""

    def __init__(self, n_genes, n_latent=10, n_hidden=128, n_batches=0, likelihood="nb"):
        super().__init__()
        self.likelihood = likelihood
        self.encoder = nn.Sequential(nn.Linear(n_genes + n_batches, n_hidden), nn.ReLU())
        self.enc_mu, self.enc_logvar = nn.Linear(n_hidden, n_latent), nn.Linear(n_hidden, n_latent)
        self.decoder = nn.Sequential(nn.Linear(n_latent + n_batches, n_hidden), nn.ReLU(),
                                     nn.Linear(n_hidden, n_genes))
        self.log_disp = nn.Parameter(torch.zeros(n_genes))     # NB: log θ_g | Gaussian: log variance of gene g

    @staticmethod
    def _join(a, b):
        return a if b is None else torch.cat([a, b], dim=1)    # append the one-hot batch, if any

    def encode(self, x_in, b=None):
        h = self.encoder(self._join(x_in, b))
        return self.enc_mu(h), self.enc_logvar(h)

    def decode(self, z, b=None):
        """Gaussian: mean log-expression. NB: proportions ρ(z) over genes (softmax, rows sum to 1)."""
        out = self.decoder(self._join(z, b))
        return torch.softmax(out, dim=1) if self.likelihood == "nb" else out

    def elbo_terms(self, x_in, x_obs, lib, b=None):
        """Per-cell reconstruction term E_q[log p(x|z)] (one sample) and KL(q(z|x) ‖ N(0, I))."""
        mu_z, logvar_z = self.encode(x_in, b)
        z = mu_z + torch.exp(0.5 * logvar_z) * torch.randn_like(mu_z)        # reparameterisation trick
        out = self.decode(z, b)
        if self.likelihood == "nb":
            rec = nb_log_prob(x_obs, lib * out, torch.exp(self.log_disp)).sum(dim=1)
        else:
            rec = -0.5 * (np.log(2 * np.pi) + self.log_disp
                          + (x_obs - out) ** 2 / torch.exp(self.log_disp)).sum(dim=1)
        kl = 0.5 * (mu_z ** 2 + torch.exp(logvar_z) - 1 - logvar_z).sum(dim=1)
        return rec, kl


demo_vae = MiniVAE(n_genes=500)
print("parameters of the NB VAE:", sum(p.numel() for p in demo_vae.parameters()))

In [ ]:
# ── 5.4 Training loop (mini-batches of cells, Adam, KL warm-up) and posterior means ──
def train_vae(model, x_in, x_obs, lib, b=None, epochs=60, batch_size=256, lr=1e-3, warmup=10, kl_weight=1.0,
              seed=SEED):
    """Maximise Σ_cells [rec − w·KL]; w rises linearly from 0 to kl_weight over `warmup` epochs (0 = no warm-up).

    Returns an (epochs × 2) array: average reconstruction term and average KL per cell in each epoch.
    """
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    data = [torch.as_tensor(a) for a in (x_in, x_obs, lib)] + ([torch.as_tensor(b)] if b is not None else [])
    history = []
    for epoch in range(epochs):
        w = kl_weight * min(1.0, (epoch + 1) / warmup) if warmup else kl_weight
        perm, rec_sum, kl_sum = torch.randperm(len(x_in)), 0.0, 0.0
        for start in range(0, len(x_in), batch_size):
            batch = [d[perm[start:start + batch_size]] for d in data]
            rec, kl = model.elbo_terms(*batch[:3], batch[3] if b is not None else None)
            loss = -(rec - w * kl).mean()                        # minimise the negative (weighted) ELBO
            opt.zero_grad(); loss.backward(); opt.step()
            rec_sum, kl_sum = rec_sum + rec.sum().item(), kl_sum + kl.sum().item()
        history.append((rec_sum / len(x_in), kl_sum / len(x_in)))
    return np.array(history)


def encode_mean(model, x_in, b=None):
    """Posterior means μ(x) of q(z|x) for all cells: the latent representation used downstream."""
    with torch.no_grad():
        mu_z, _ = model.encode(torch.as_tensor(x_in), None if b is None else torch.as_tensor(b))
    return mu_z.numpy()

**KL warm-up.** In the first epochs the decoder is still random, and a full-strength KL term can push every code to the
prior before the decoder has learned to use them (*posterior collapse*, Section 8). A common remedy is to multiply the KL
by a weight that rises from 0 to 1 over the first epochs (Bowman *et al.* 2016); scVI uses a similar schedule. We use 10
epochs of warm-up. Now we train both VAEs (about 10–30 s on a CPU).

In [ ]:
# ── 5.5 Train a Gaussian VAE (log-normalised data) and an NB VAE (raw counts = mini-scVI) ──
t0 = time.time()
torch.manual_seed(SEED)
vae_gauss = MiniVAE(n_genes=500, likelihood="gaussian")
hist_gauss = train_vae(vae_gauss, lognorm, lognorm, lib_size, epochs=60)
t1 = time.time()
torch.manual_seed(SEED)
vae_nb = MiniVAE(n_genes=500, likelihood="nb")
hist_nb = train_vae(vae_nb, lognorm, counts, lib_size, epochs=60)
print(f"Gaussian VAE: {t1 - t0:.0f} s | NB VAE: {time.time() - t1:.0f} s  (60 epochs × 12 mini-batches each)")
print(f"NB VAE per cell, epoch 1 → 60: reconstruction {hist_nb[0, 0]:.1f} → {hist_nb[-1, 0]:.1f} nats, "
      f"KL {hist_nb[0, 1]:.1f} → {hist_nb[-1, 1]:.1f} nats")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
a1.plot(np.arange(1, 61), hist_nb[:, 0], color="tab:blue")
a1.set_xlabel("epoch"); a1.set_ylabel("E_q[log p(x|z)] per cell (nats)")
a1.set_title("NB VAE: reconstruction term rises")
a2.plot(np.arange(1, 61), hist_nb[:, 1], color="tab:orange")
a2.axvspan(1, 10, color="0.9", label="KL warm-up (weight 0 → 1)")
a2.set_xlabel("epoch"); a2.set_ylabel("KL(q(z|x) ‖ N(0, I)) per cell (nats)"); a2.legend(fontsize=7)
a2.set_title("KL term: overshoots during warm-up, then settles")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both models trained in seconds. For the NB VAE the reconstruction term (the expected
log-probability of a cell's 500 counts) rose steeply in the first few epochs and then slowly; the KL term peaked while
its weight was still small — the encoder spreads codes out freely — and was pulled back once the weight reached 1 at
epoch 10, ending at a few nats per cell: that is how much information about each cell the 10 latent numbers carry
beyond the prior.

⚠️ **Do not compare the two ELBOs.** The Gaussian VAE models continuous log-normalised values and the NB VAE models
integer counts; their log-likelihoods live on different scales (a density vs a probability) and are not comparable
numbers. Compare models by what they are for — the latent space and the predictions — as we do next.

✏️ **Your turn.** Train the NB VAE with `n_latent=2` and plot `encode_mean(...)` directly with `plot_embedding`. Which cell
types can two latent numbers separate, and which merge?

---
## 6 · 🧪 Is the latent space biologically meaningful?

> 📍 **Where we are.** We have three 10-dimensional representations of the same 3,000 cells: 10 principal components of
> the log-normalised data ([Module 14](14_pca_dimensionality_reduction.ipynb), the standard baseline), the Gaussian VAE's
> posterior means and the NB VAE's posterior means. Because the data are simulated, we know every cell's true type.

Three ground-truth scores, each asking a slightly different question
([Modules 08](08_knn_naive_bayes.ipynb), [12](12_clustering_kmeans_hierarchical.ipynb), [13](13_mixture_models_em.ipynb)):

- **k-NN accuracy** (5-fold cross-validated, $k=15$): are a cell's neighbours of its own type? (local structure)
- **silhouette** of the true types: are types compact and well separated? (from −1 to 1)
- **adjusted Rand index (ARI)** of k-means with $k=6$ against the truth: would an unsupervised clustering recover the types?

In [ ]:
# ── 6.1 Three ground-truth scores for three representations ──
def latent_scores(Z, labels, seed=SEED):
    """k-NN accuracy (5-fold CV), silhouette of the true labels, and ARI of k-means against the truth."""
    knn = cross_val_score(KNeighborsClassifier(n_neighbors=15), Z, labels, cv=5).mean()
    sil = silhouette_score(Z, labels)
    km = KMeans(n_clusters=len(np.unique(labels)), n_init=10, random_state=seed).fit_predict(Z)
    return {"kNN accuracy": knn, "silhouette": sil, "k-means ARI": adjusted_rand_score(labels, km)}


Z_pca, _, _ = pca(lognorm, 10)
Z_gauss, Z_nb = encode_mean(vae_gauss, lognorm), encode_mean(vae_nb, lognorm)
reps = {"PCA (10 PCs)": Z_pca, "Gaussian VAE (10-D)": Z_gauss, "NB VAE (10-D)": Z_nb}
score_table = pd.DataFrame({name: latent_scores(Z, cell_type) for name, Z in reps.items()}).T
print(score_table.round(3).to_string())

🔍 **What just happened?** All three representations separate the six simulated types well — this toy is
PCA-friendly, because the simulator builds types from additive shifts on the log scale, which is nearly what PCA
assumes. Still, the NB VAE has the clearest structure: the highest silhouette (0.44 vs 0.31 for PCA) and the highest k-NN
accuracy (0.996) and k-means ARI (0.986). The Gaussian VAE is on a par with PCA (slightly higher silhouette, slightly
lower accuracy and ARI). An honest
summary: on easy data a good linear baseline is hard to beat (benchmarks on real data reach similar conclusions for
simple tasks), and a count likelihood helps when noise matters. Let us *look* at the spaces too.

In [ ]:
# ── 6.2 2-D views (first two principal components of each representation) ──
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (name, Z) in zip(axes[:3], reps.items()):
    plot_embedding(pca(Z, 2)[0], cell_type, ax=ax, title=name, legend=(ax is axes[0]))
plot_embedding(pca(Z_nb, 2)[0], condition, ax=axes[3], title="NB VAE, coloured by condition")
fig.suptitle("SIMULATED PBMC-like cells: true types (first three panels) and stimulation (last)", fontsize=10)
plt.tight_layout(); plt.show()
for name, Z in reps.items():                     # 🧪 is stimulation (never shown to any model) encoded too?
    acc_cond = cross_val_score(KNeighborsClassifier(n_neighbors=15), Z, condition, cv=5).mean()
    print(f"{name:20s}: k-NN accuracy for control vs stimulated = {acc_cond:.3f}  (chance ≈ 0.5)")

🔍 **What just happened?** No 2-D view separates all six types: the first two directions of each space capture the largest
differences (monocytes and dendritic cells apart from lymphocytes; in the two VAE spaces also B cells), while CD4 T, CD8 T
and NK cells overlap. The 10-D spaces do separate them, as the scores in 6.1 show — a 2-D picture shows only part of what
a representation contains. The same holds for the stimulation state: in the last panel control and stimulated cells
overlap in 2-D, yet a k-NN classifier on any of the three 10-D spaces predicts the condition with about 81–83 % accuracy
(chance ≈ 50 %). The representations encode **state** as well as **type**, without having been told about either.

⚠️ **Pictures are not evidence.** 2-D projections (and t-SNE/UMAP maps) can exaggerate or hide structure; always back
claims with metrics against ground truth or biology, as in 6.1.

**Posterior predictive check.** A generative model makes a stronger promise than a good embedding: data simulated from the
fitted model should look like the real data. For each cell we draw $z\sim q_\phi(z\mid x)$, decode, and sample a new count
profile — from the NB decoder directly, and from the Gaussian decoder by sampling log-normalised values and inverting the
normalisation with each cell's total count over all 1,000 genes (the total `log_normalize` used), rounding to
non-negative integers. Then we compare per-gene zero fractions and the mean–variance relation
with the real counts. As a second 🧪 check, the simulator's **true** gene dispersions $\theta_g$ are known.

In [ ]:
# ── 6.3 🧪 Posterior predictive check: do simulated counts look like the real ones? ──
gen = np.random.default_rng(SEED + 63)
torch.manual_seed(SEED + 63)
with torch.no_grad():
    x_in = torch.as_tensor(lognorm)
    mu_z, logvar_z = vae_nb.encode(x_in)
    rho = vae_nb.decode(mu_z + torch.exp(0.5 * logvar_z) * torch.randn_like(mu_z)).numpy()
    theta_hat = torch.exp(vae_nb.log_disp).numpy()
    mu_g, logvar_g = vae_gauss.encode(x_in)
    m_gauss = vae_gauss.decode(mu_g + torch.exp(0.5 * logvar_g) * torch.randn_like(mu_g)).numpy()
    sd_gauss = torch.exp(0.5 * vae_gauss.log_disp).numpy()
sim_nb = sample_negative_binomial(gen, lib_size * rho, theta_hat)                  # NB decoder → counts
sim_log = m_gauss + sd_gauss * gen.standard_normal(m_gauss.shape)                    # Gaussian decoder → log data
lib_all = sc["counts"].sum(axis=1, keepdims=True)                                  # total over ALL 1,000 genes,
sim_gauss = np.clip(np.round(lib_all * np.expm1(sim_log) / 1e4), 0, None)           # as used by log_normalize
zero_real = (counts == 0).mean(axis=0)
for name, sim in [("NB decoder", sim_nb), ("Gaussian decoder", sim_gauss)]:
    err = np.abs((sim == 0).mean(axis=0) - zero_real).mean()
    print(f"{name:16s}: overall zeros {100 * (sim == 0).mean():.1f} % (real {100 * (counts == 0).mean():.1f} %), "
          f"mean per-gene |zero-fraction error| = {err:.3f}, total UMIs {sim.sum() / counts.sum():.2f} × real")
gene_mean = counts.mean(axis=0)
bins = {"mean < 0.5": gene_mean < 0.5, "0.5–2": (gene_mean >= 0.5) & (gene_mean < 2), "≥ 2": gene_mean >= 2}
for name, sim in [("NB decoder", sim_nb), ("Gaussian decoder", sim_gauss)]:
    ratio = sim.var(axis=0) / counts.var(axis=0)                                     # per-gene variance, model / real
    print(f"{name:16s}: median variance ratio (model / real) by mean count: "
          + ", ".join(f"{k} UMIs ({m.sum()} genes) {np.median(ratio[m]):.2f}" for k, m in bins.items()))
theta_true = sc["var"]["theta"].to_numpy()[hvg]
print(f"gene dispersions θ: learned median {np.median(theta_hat):.2f} vs true median {np.median(theta_true):.2f}; "
      f"Spearman ρ(learned, true) = {stats.spearmanr(theta_hat, theta_true)[0]:.2f}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.scatter(zero_real, (sim_nb == 0).mean(0), s=5, color="tab:blue", label="NB decoder")
a1.scatter(zero_real, (sim_gauss == 0).mean(0), s=5, color="tab:orange", label="Gaussian decoder")
a1.plot([0, 1], [0, 1], "k--", lw=0.8); a1.set_xlabel("fraction of zeros per gene, real (SIMULATED) data")
a1.set_ylabel("fraction of zeros, model samples"); a1.legend(fontsize=7); a1.set_title("Zeros: NB on the diagonal")
for name, A, col in [("real", counts, "0.3"), ("NB decoder", sim_nb, "tab:blue"), ("Gaussian decoder", sim_gauss, "tab:orange")]:
    a2.scatter(A.mean(0), A.var(0), s=4, color=col, label=name, alpha=0.7)
a2.set_xscale("log"); a2.set_yscale("log"); a2.set_xlabel("mean count per gene"); a2.set_ylabel("variance per gene")
a2.legend(fontsize=7); a2.set_title("Mean–variance relation of counts")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Counts sampled from the NB decoder reproduce the real per-gene zero fractions closely (points
near the diagonal; mean absolute error 0.033; 61.5 % zeros overall against 58.2 %) and the right total number of UMIs
(1.00×). Their variances are too large, increasingly so for more highly expressed genes: the median per-gene ratio of
model to real variance is 1.19 for genes with fewer than 0.5 UMIs per cell, 1.36 for 0.5–2 and 1.67 above 2 — a first
hint of the dispersion issue below. The Gaussian decoder, which never saw counts, does worse on every check: too many
zeros (70.0 % overall; per-gene error 0.118, more than three times the NB decoder's), 12 % too many UMIs, and a
mean–variance relation of the wrong shape — variances too small for rare genes (median ratio 0.69) and far too large for
moderately and highly expressed ones (4.3 and 7.5). Symmetric noise in log space, mapped back through the exponential,
cannot represent the discrete, skewed distribution of small counts. **The likelihood is part of the model's biology**: it encodes what we know
about the assay (Module 24).

The dispersion check is humbling in an instructive way: the learned $\theta_g$ are about 5 times **smaller** than the true
ones (median 1.7 vs 7.8, i.e. more overdispersion than simulated), and only weakly ordered like the truth (Spearman 0.23). The NB decoder must attribute to
"noise" everything its 10-D latent code and decoder do not explain — donor effects, imperfectly encoded states and the
randomness of $z$ itself during training — so $\theta_g$ is an *effective* dispersion of the model, not a measurement of
the gene's biological noise. Parameters of a fitted deep generative model should be interpreted with that in mind.

✏️ **Your turn.** A posterior predictive check can target any statistic you care about. Find a monocyte marker among the
500 genes (`genes = sc["var"].index[hvg]`; try `"LYZ"` or `"S100A9"`) and compare the histogram of its counts in real
monocytes with the counts sampled from the NB and from the Gaussian decoder for the same cells. Which decoder reproduces
the spread of a highly expressed marker?

### 📄 Research connection — Deep generative models of single cells

- **What the field does.** **scVI** (Lopez *et al.* 2018) introduced the recipe built here — a VAE with a (zero-inflated)
    negative-binomial likelihood for raw UMI counts, library size and batch in the model, trained by stochastic VI on
    mini-batches — and showed that it scales to more than a million cells. The **scvi-tools** library (Gayoso *et al.*
    2022) now hosts a family of such models: **scANVI** adds cell-type labels for semi-supervised annotation and label
    transfer (Xu *et al.* 2021), **totalVI** models RNA and surface proteins jointly (Gayoso *et al.* 2021), and further
    models cover chromatin accessibility and spatial data. They are used for integration, annotation, differential
    expression with uncertainty and reference mapping.
- **What this notebook builds.** A mini-scVI: Gaussian encoder, softmax decoder for expression proportions, observed library size, learned
    gene-wise NB dispersion, KL warm-up, one reparameterised sample per cell; evaluated against true cell types and with a
    posterior predictive check. Production scVI differs mainly in scale (thousands of genes, deeper networks, GPUs),
    options (zero inflation, latent library size, more covariates) and engineering (data loaders, early stopping).
- **Where the frontier is.** Active questions (as of 2025): how to evaluate latent spaces fairly (benchmarks find that simple PCA-based
    pipelines remain competitive on many tasks); disentangling biological factors (type, state, perturbation, donor) in
    the latent space; uncertainty-aware downstream analyses; and **single-cell foundation models** pre-trained on tens of
    millions of cells, whose advantages over task-specific models such as scVI are still being assessed (Module 43).
- **Read next.** Lopez *et al.* (2018) *Deep generative modeling for single-cell transcriptomics*; Gayoso *et al.* (2022)
    *A Python library for probabilistic analysis of single-cell omics data*; Xu *et al.* (2021) *Probabilistic harmonization
    and annotation of single-cell transcriptomics data with deep generative models* (scANVI).

---
## 7 · Conditioning on batch

> 📍 **Where we are.** Real data sets are assembled from several **batches** — samples processed on different days, with
> different reagent lots, or in different labs. Batch effects shift expression systematically, and an unsupervised model
> will happily spend its latent dimensions describing them. Here we give the VAE the batch label as an input, the key idea
> of scVI's integration, and test it against the truth. [Module 31](31_batch_integration.ipynb) compares this with
> other integration methods.

**Why conditioning helps.** In the generative model, the batch $b_c$ of each cell is *observed*: $x_c\sim p_\theta(x\mid
z_c,b_c)$. If the decoder can explain batch-specific shifts from $b_c$ directly, the latent $z_c$ no longer needs to encode
them — and the KL term actively discourages $z$ from carrying information the decoder can get for free. So $z$ becomes
(approximately) a batch-free description of the cell's biology. The encoder also receives $b_c$, so that it can "subtract"
the batch when inferring $z$.

> 🔗 **Builds on [Module 06](06_linear_regression_gwas.ipynb).** Giving the decoder the batch label is the neural
> version of adding a batch column to a regression design matrix: the other effects are estimated *adjusting for* batch.
> Like any covariate adjustment, it can only work if batch is not confounded with what you want to keep.

To measure success we need two numbers, because removing batch effects is easy if we are allowed to destroy biology:

- **batch-mixing entropy**: for each cell, the entropy of the batch labels among its $k=30$ nearest neighbours in the latent
  space, divided by $\log(\text{number of batches})$ — 0 if all neighbours come from the cell's own batch, 1 if the
  batches are perfectly mixed (with balanced batches);
- **cell-type k-NN accuracy** as before: biology must be preserved.

In [ ]:
# ── 7.1 SIMULATED data with two batches and a strong batch effect ──
rng = np.random.default_rng(SEED + 8)
sc2 = simulate_scrna(rng, n_cells=3000, n_genes=1000, n_batches=2, batch_effect=0.5, depth=800)
obs2 = sc2["obs"]
lognorm2_all = log_normalize(sc2["counts"])
hvg2 = np.argsort(lognorm2_all.var(axis=0))[::-1][:500]
counts2, lognorm2 = sc2["counts"][:, hvg2].astype(np.float32), lognorm2_all[:, hvg2].astype(np.float32)
lib2 = counts2.sum(axis=1, keepdims=True)
type2, batch2 = obs2["true_type"].to_numpy(str), obs2["batch"].to_numpy(str)
batch_onehot = (batch2[:, None] == np.array(["batch1", "batch2"])).astype(np.float32)   # cells × 2
print(pd.crosstab(obs2["batch"], obs2["condition"]).to_string())
print("each batch contains all six types:", pd.crosstab(type2, batch2).min().min() > 0)


def batch_mixing_entropy(Z, batches, k=30):
    """Mean normalised entropy of batch labels among each cell's k nearest neighbours (0 = separated, 1 = mixed)."""
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
    levels = np.unique(batches)
    frac = np.stack([(batches[idx] == lev).mean(axis=1) for lev in levels], axis=1)   # cells × batches
    ent = -np.sum(np.where(frac > 0, frac * np.log(np.where(frac > 0, frac, 1)), 0), axis=1)
    return float(ent.mean() / np.log(len(levels)))

In [ ]:
# ── 7.2 Train an ordinary NB VAE and a batch-conditioned NB VAE; score mixing and conservation ──
t0 = time.time()
torch.manual_seed(SEED)
vae_plain = MiniVAE(n_genes=500)
hist_plain = train_vae(vae_plain, lognorm2, counts2, lib2, epochs=50)
torch.manual_seed(SEED)
vae_batch = MiniVAE(n_genes=500, n_batches=2)
hist_batch = train_vae(vae_batch, lognorm2, counts2, lib2, b=batch_onehot, epochs=50)
print(f"two VAEs trained in {time.time() - t0:.0f} s")
Z2 = {"PCA (10 PCs)": pca(lognorm2, 10)[0], "NB VAE": encode_mean(vae_plain, lognorm2),
      "batch-conditioned NB VAE": encode_mean(vae_batch, lognorm2, batch_onehot)}
mix_table = pd.DataFrame({name: {"batch-mixing entropy": batch_mixing_entropy(Z, batch2),
                                 "cell-type kNN accuracy": latent_scores(Z, type2)["kNN accuracy"]}
                          for name, Z in Z2.items()}).T
print(mix_table.round(3).to_string())

In [ ]:
# ── 7.3 Look: coloured by batch (top) and by true type (bottom) ──
fig, axes = plt.subplots(2, 3, figsize=(11, 6.2))
for col, (name, Z) in enumerate(Z2.items()):
    xy = pca(Z, 2)[0]
    plot_embedding(xy, batch2, ax=axes[0, col], title=f"{name}: batch", legend=(col == 0))
    plot_embedding(xy, type2, ax=axes[1, col], title=f"{name}: true type", legend=(col == 0))
fig.suptitle("SIMULATED two-batch data: only the batch-conditioned VAE mixes batches while keeping types apart",
             fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In PCA and in the ordinary VAE, every cell type appears **twice**, once per batch: the batch
effect is as large as the differences between some types, and the batch-mixing entropy is essentially 0 (neighbours
come from the cell's own batch). The batch-conditioned VAE produces one island per type with the two batches
interleaved (entropy 0.88), while k-NN cell-type accuracy is unchanged (0.968–0.969 for all three representations). The
decoder now explains the batch shift, so the latent code does not have to.

🧪 **Counterfactual decoding.** Because the decoder takes the batch as an input, we can ask: *what would this batch-1 cell
look like had it been sequenced in batch 2?* Encode it with its own batch, then decode the same $z$ with the other batch
label. The simulation lets us check the answer: compare the counterfactual average profile of batch-1 B cells with the
**real** batch-2 B cells.

In [ ]:
# ── 7.4 🧪 Counterfactual: decode batch-1 cells as if they came from batch 2 ──
with torch.no_grad():
    z_b = torch.as_tensor(Z2["batch-conditioned NB VAE"])
    rho_own = vae_batch.decode(z_b, torch.as_tensor(batch_onehot)).numpy()             # as observed
    rho_swap = vae_batch.decode(z_b, torch.as_tensor(batch_onehot[:, ::-1].copy())).numpy()   # other batch
frac2 = counts2 / lib2                                                                   # observed proportions
print("type      | corr(counterfactual, real batch 2) | corr(original, real batch 2) | corr(pred. vs obs. batch effect)")
for t in ["B", "Monocyte", "CD4 T", "NK"]:
    in1, in2 = (type2 == t) & (batch2 == "batch1"), (type2 == t) & (batch2 == "batch2")
    expressed = frac2[in1 | in2].mean(axis=0) > 1e-3                                     # skip genes with ~no counts
    real1, real2 = np.log(frac2[in1].mean(0) + 1e-6), np.log(frac2[in2].mean(0) + 1e-6)
    cf, own = np.log(rho_swap[in1].mean(0) + 1e-6), np.log(rho_own[in1].mean(0) + 1e-6)
    r_cf, r_orig = np.corrcoef(cf[expressed], real2[expressed])[0, 1], np.corrcoef(real1[expressed], real2[expressed])[0, 1]
    r_eff = np.corrcoef((cf - own)[expressed], (real2 - real1)[expressed])[0, 1]
    print(f"{t:9s} |              {r_cf:.3f}                |            {r_orig:.3f}           |        {r_eff:.3f}")
    if t == "B":
        eff_pred, eff_obs = (cf - own)[expressed], (real2 - real1)[expressed]
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.scatter(eff_obs, eff_pred, s=6, color="tab:purple")
ax.plot([-2, 2], [-2, 2], "k--", lw=0.8)
ax.set_xlabel("observed batch effect in B cells\n(log ratio, real batch 2 vs batch 1)")
ax.set_ylabel("predicted by counterfactual decoding")
ax.set_title("Per-gene batch effect: prediction vs truth (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For every cell type tested, the counterfactual profile of batch-1 cells decoded "as batch 2"
correlates with the real batch-2 cells of that type far better (0.991–0.996) than the original batch-1 profile does
(0.71–0.76), and the per-gene batch effect it predicts agrees with the observed one (correlation 0.987–0.992; points on
the diagonal). On this simulation, where the batch effect is a gene-specific shift shared by all types, the model has
learned it correctly.

⚠️ **A counterfactual is only as good as the model's assumptions.** Batch here is a technical factor that was
not confounded with biology — every cell type occurs in both batches, and so do both conditions (unequally: 978 control
and 482 stimulated cells in batch 1, 486 and 1,054 in batch 2) — so the model could separate "batch" from "biology".
If a cell type existed in only one batch, or if all stimulated donors were in one batch, the same decoding would invent
expression profiles never observed (Module 31 provokes exactly this). And "what would this cell look like *if treated*"
is a **causal** question that latent arithmetic cannot answer without causal assumptions (Module 41).

✏️ **Your turn.** Use `rho_swap` to compute, for each gene, the mean predicted log-ratio (batch 2 vs batch 1) over *all*
cells, and plot it against the same quantity computed separately for monocytes and for T cells. Is the learned batch effect
the same for all types? Is that what the simulator assumes (look at `batch_shift` in `simulate_scrna`)?

### 📄 Research connection — Latent spaces for prediction — and their caveats

- **What the field does.** Once cells live in a smooth latent space, it is tempting to *predict* by moving through it. **scGen**
    (Lotfollahi, Wolf & Theis 2019) predicted a cell type's response to a perturbation by adding a "perturbation vector"
    (the latent difference between stimulated and control cells of other types) — latent-vector arithmetic, a cousin of
    the decoder-side label swap we used for batches, where $z$ stays fixed and only the covariate fed to the decoder changes.
    The **compositional perturbation autoencoder**, CPA (Lotfollahi *et al.* 2023), learns separate latent embeddings for
    basal state, perturbations, doses and covariates and recombines them to predict unseen combinations.
- **What this notebook builds.** Counterfactual decoding with a batch-conditioned decoder, checked against the true batch-2 profiles of the
    same cell types. This is the simplest version of covariate swapping: the covariate enters the decoder, and changing it
    while holding $z$ fixed gives a prediction.
- **Where the frontier is.** Whether latent arithmetic generalises to new cell types, strong perturbations or combinations is an open
    question; recent benchmarks of perturbation-response prediction (2024–2025) report that simple baselines (e.g.
    predicting the average response) are often hard to beat. Latent shifts are associations learned from the training
    distribution, not causal mechanisms — [Module 41](41_perturbation_causal_inference.ipynb) discusses what causal claims
    require.
- **Read next.** Lotfollahi, Wolf & Theis (2019) *scGen predicts single-cell perturbation responses*; Lotfollahi *et al.* (2023)
    *Predicting cellular responses to complex perturbations in high-throughput screens* (CPA).

---
## 8 · Pitfalls: posterior collapse, latent dimension and identifiability

> 📍 **Where we are.** The VAE worked well with sensible settings. Here we break it on purpose, to recognise the failure
> modes in real analyses.

**Posterior collapse.** The ELBO trades reconstruction against $\mathrm{KL}(q(z\mid x)\,\|\,p(z))$. If the KL is too
expensive — because the KL term is up-weighted, or the decoder can explain the data without $z$ — the optimum is
$q(z\mid x)=p(z)$ for every cell: the code carries **no information**, yet the model still "reconstructs" every cell by its
library size times the average profile. A **β-VAE** (Higgins *et al.* 2017) multiplies the KL by a weight $\beta$; $\beta>1$
encourages simpler, sometimes more interpretable codes, at the risk of collapse. A latent dimension counts as **active**
if its posterior mean varies across cells, $\mathrm{Var}_{x}\big(\mathbb E_q[z_k\mid x]\big)>0.01$ (Burda *et al.* 2016).
We retrain the NB VAE of Section 5 with $\beta=5$ and $\beta=20$ (no warm-up) and compare with $\beta=1$.

In [ ]:
# ── 8.1 Provoke posterior collapse by up-weighting the KL term (β-VAE) ──
def collapse_report(model, name):
    """Active dimensions, KL per dimension, reconstruction and cell-type kNN accuracy of a trained NB VAE."""
    with torch.no_grad():
        mu_z, logvar_z = model.encode(torch.as_tensor(lognorm))
        rec, _ = model.elbo_terms(torch.as_tensor(lognorm), torch.as_tensor(counts), torch.as_tensor(lib_size))
    kl_dim = (0.5 * (mu_z ** 2 + logvar_z.exp() - 1 - logvar_z)).mean(0).numpy()
    active = int((mu_z.numpy().var(axis=0) > 0.01).sum())
    acc = latent_scores(mu_z.numpy(), cell_type)["kNN accuracy"]
    print(f"{name:13s}: active dims {active:2d}/10 | KL {kl_dim.sum():5.2f} nats | reconstruction "
          f"{rec.mean().item():7.1f} nats | kNN accuracy {acc:.3f}")
    return np.sort(kl_dim)[::-1]


t0 = time.time()
kl_profiles, kl_traces = {"β = 1": collapse_report(vae_nb, "β = 1 (Sec. 5)")}, {"β = 1": hist_nb[:40, 1]}
for beta in (5.0, 20.0):
    torch.manual_seed(SEED)
    model = MiniVAE(n_genes=500)
    hist = train_vae(model, lognorm, counts, lib_size, epochs=40, warmup=0, kl_weight=beta)
    kl_profiles[f"β = {beta:.0f}"] = collapse_report(model, f"β = {beta:.0f}")
    kl_traces[f"β = {beta:.0f}"] = hist[:, 1]
print(f"(two extra VAEs trained in {time.time() - t0:.0f} s)")

In [ ]:
# ── 8.2 Picture: KL during training and KL carried by each latent dimension ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
colors = {"β = 1": "tab:blue", "β = 5": "tab:orange", "β = 20": "tab:red"}
for name, tr in kl_traces.items():
    a1.plot(np.arange(1, len(tr) + 1), tr, color=colors[name], label=name)
a1.set_xlabel("epoch"); a1.set_ylabel("KL per cell (nats)"); a1.set_yscale("symlog", linthresh=0.1)
a1.set_title("KL term during training"); a1.legend(fontsize=7)
width = 0.27
for k, (name, prof) in enumerate(kl_profiles.items()):
    a2.bar(np.arange(1, 11) + (k - 1) * width, prof, width=width, color=colors[name], label=name)
a2.set_xlabel("latent dimension (sorted by KL)"); a2.set_ylabel("average KL of this dimension (nats)")
a2.set_title("Information carried by each latent dimension"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With $\beta=1$ all ten dimensions are active and together carry 5.0 nats per cell, spread
unevenly (the first five carry most). With $\beta=5$ the total KL falls tenfold, to 0.51 nats; five dimensions still vary
across cells but carry so little information that cell-type accuracy drops to 0.67. With $\beta=20$ the posterior
collapses completely: no active dimension, KL ≈ 0, and k-NN accuracy 0.35 (chance is about 0.17; what remains is tiny,
sub-threshold variation of the posterior means). Yet the reconstruction term is only about 4 % worse (−545.7 vs −523.6
nats per cell), because library size times the average expression profile already explains most of each cell's counts. **A collapsed VAE does not crash and its loss
looks reasonable**; only the latent diagnostics reveal it.

**Remedies** (when collapse is unwanted): KL warm-up, as in Section 5 (Bowman *et al.* 2016); a weaker decoder; "free bits"
(a floor on the KL per dimension); and checking active dimensions routinely.

**Choosing the latent dimension.** There is no single right value. Too few dimensions merge biology (try ✏️ in Section 5
with 2); too many are harmless but unused — the KL term switches off dimensions the data do not need, as the uneven bars
show. scVI's default is 10; atlas-scale models often use 10–30. Choose by downstream metrics on held-out data or known
biology, not by the ELBO alone.

**Identifiability.** The prior $\mathcal N(0,I)$ is unchanged by rotating $z$, so a decoder composed with any rotation gives
the same likelihood: individual latent axes have **no fixed meaning**, and two training runs (or two models) can produce
different axes for the same biology. Let us check by comparing the Gaussian-VAE and NB-VAE latent spaces of Section 5.

In [ ]:
# ── 8.3 Same geometry, different axes: compare two latent spaces of the same cells ──
Zg_c, Zn_c = Z_gauss - Z_gauss.mean(0), Z_nb - Z_nb.mean(0)
coef, *_ = np.linalg.lstsq(Zg_c, Zn_c, rcond=None)                    # best linear map Gaussian-VAE → NB-VAE space
r2 = 1 - np.sum((Zn_c - Zg_c @ coef) ** 2) / np.sum(Zn_c ** 2)
cross = np.corrcoef(Zg_c.T, Zn_c.T)[:10, 10:]                          # axis-by-axis correlations
print(f"a linear map explains R² = {r2:.2f} of the NB-VAE latent space from the Gaussian-VAE one")
print("for each NB-VAE axis, its best |correlation| with any single Gaussian-VAE axis:",
      np.round(np.abs(cross).max(axis=0), 2))

🔍 **What just happened?** A single linear map predicts most of the NB-VAE latent coordinates from the Gaussian-VAE ones
(R² printed above): the two models found largely the same *geometry* of cell states. But no axis of one model corresponds
cleanly to an axis of the other (best single-axis correlations well below 1). **Interpret distances and neighbourhoods in a
latent space, not individual coordinates** — unless the model was built to make axes meaningful (e.g. a linear decoder
with sparsity, or supervised factors).

⚠️ **Evaluation needs ground truth or biology.** Everything in Sections 6–8 was checked against simulated truth. On real
data, use known cell types, held-out marker genes, replicate donors, or spike-in batches — and be suspicious of a model
judged only by its loss or by how pretty its 2-D map looks.

✏️ **Your turn.** Retrain the NB VAE of Section 5 with a different seed (`torch.manual_seed(1)` before construction and
`seed=1` in `train_vae`). Compare its latent space with `Z_nb` using the code of 8.3, and its cell-type scores with 6.1.
Which quantities are reproducible across seeds and which are not?

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran on 3,000 simulated cells and 500 genes in about a minute. This section is
> reading: what changes when you apply the same ideas to real atlases.

**Scale and compute.** Real studies range from $10^4$ to millions of cells and use 2,000–5,000 highly variable genes. scVI-type
models train with the same mini-batch SGD you wrote, typically on a GPU, in minutes to an hour; the encoder then maps new
cells in one forward pass, which is what makes reference mapping (Module 31) fast.

**Public data and resources.**
[CELLxGENE](https://cellxgene.cziscience.com/) (curated single-cell data sets and the Census API to query tens of millions
of cells), the [Human Cell Atlas](https://www.humancellatlas.org/) data portal, and the PBMC data sets distributed by
[10x Genomics](https://www.10xgenomics.com/datasets) — including PBMC experiments like the interferon-β stimulation of
Kang *et al.* (2018) that our simulation mimics.

**Production tools.** [scvi-tools](https://scvi-tools.org/) (scVI, scANVI, totalVI and many more; works with AnnData and
Scanpy), [Pyro](https://pyro.ai/) and NumPyro for writing your own VI models, and Stan's ADVI for automatic VI on
statistical models. In scvi-tools the steps of this notebook map onto `setup_anndata` (choose counts, batch key), model
construction (latent dimension, gene likelihood `nb`/`zinb`), `train()` (with KL warm-up and early stopping), and
`get_latent_representation()`.

**What changes with real data — the first three things that go wrong.**

1. **Hidden technical structure.** Real batches are not a clean per-gene shift; donors, chemistries and dissociation
   protocols are nested and partly confounded with biology. Conditioning on the wrong covariate (or on one confounded with
   the condition of interest) removes real biology — see [Module 31](31_batch_integration.ipynb).
2. **Artefact cells become latent structure.** Doublets, damaged cells and ambient RNA form their own islands in the latent
   space unless quality control removes them first ([Module 30](30_single_cell_rnaseq.ipynb)).
3. **Results depend on settings and seeds.** Latent dimension, number of genes, epochs and the random seed change the
   embedding; report them, check robustness, and evaluate with biological ground truth rather than with the ELBO or a UMAP.

A modelling note: the original scVI used a zero-inflated NB; for UMI data a plain NB is usually adequate (Svensson 2020),
and scvi-tools lets you choose. **VI beyond single cells:** variational methods and VAEs are used across computational
biology, e.g. to model continuous conformational heterogeneity in cryo-EM (cryoDRGN; Zhong *et al.* 2021) — the sister
course's [BiologyToResearch Module 23](../../../../cryoem/tutorial/BiologyToResearch/notebooks/23_disentangling_identity_pose.ipynb) builds VAEs for molecular images, and
[Module 27](../../../../cryoem/tutorial/BiologyToResearch/notebooks/27_cryodrgn_neural_fields.ipynb) builds a toy cryoDRGN.

---
## 📌 Key takeaways

- **Approximate inference is needed** whenever $p(x)=\int p(x\mid z)p(z)\,dz$ has no closed form — e.g. neural-network
  likelihoods for counts. EM needs the exact posterior; MCMC is exact but slow; **VI** turns inference into optimisation.
- **KL divergence** is non-negative and asymmetric; VI minimises $\mathrm{KL}(q\,\|\,p)$, which is **mode-seeking** and
  tends to **underestimate uncertainty**.
- **$\log p(x)=\mathrm{ELBO}(q)+\mathrm{KL}(q\,\|\,p(z\mid x))$**: maximising the ELBO is minimising the KL to the
  posterior without computing it; **EM is coordinate ascent on the ELBO** with an exact E-step.
- **Mean-field CAVI** gets posterior means right but variances too small and dependence wrong (≈ 5 % too narrow for one
  gene); for linked SNPs in logistic regression mean-field VI was about 4× too narrow — enough to name the wrong variant.
- The **reparameterisation trick** $z=\mu+\sigma\varepsilon$ gives low-variance gradients (about 50× lower variance than
  the score-function estimator here) and makes VI work with any differentiable model via autograd.
- A **VAE** = amortised VI (encoder) + a neural generative model (decoder), trained on the ELBO
  "reconstruction − KL". With a **negative-binomial decoder, observed library size and batch input** it is a mini-scVI:
  its latent space separated the true types, its posterior predictive matched the zeros far better than a Gaussian
  decoder's (though with somewhat too large variances), and batch conditioning mixed
  batches while preserving types.
- **Pitfalls:** posterior collapse (a large KL weight gave zero active dimensions with a reasonable-looking loss), latent
  dimensions chosen by downstream metrics, non-identifiable axes, and evaluation that must use ground truth or biology.

---
## ✅ Checkpoint

1. Why does minimising $\mathrm{KL}(q\,\|\,p)$ tend to underestimate posterior variance, whereas $\mathrm{KL}(p\,\|\,q)$ overestimates it?
   <details><summary>Answer</summary>$\mathrm{KL}(q\,\|\,p)=\mathbb E_q[\log q-\log p]$ is huge wherever $q$ puts mass and $p$ is tiny, so $q$ retreats into the high-probability region of $p$ (one mode, narrow). $\mathrm{KL}(p\,\|\,q)$ averages under $p$ and is huge where $p$ has mass but $q$ is tiny, so $q$ spreads to cover all of $p$ — for a single Gaussian it matches $p$'s mean and variance (Section 1: N(−2, 0.6²) vs N(0, 2.1²)).</details>
2. Starting from $\log p(x)=\mathbb E_q[\log p(x)]$, derive $\log p(x)=\mathrm{ELBO}(q)+\mathrm{KL}(q\,\|\,p(z\mid x))$. What are the ELBO and the KL when $q$ is the exact posterior?
   <details><summary>Answer</summary>Write $p(x)=p(x,z)/p(z\mid x)$ and multiply and divide by $q(z)$: $\mathbb E_q[\log\frac{p(x,z)}{q(z)}]+\mathbb E_q[\log\frac{q(z)}{p(z\mid x)}]$. For $q=p(z\mid x)$ the KL is 0 and the ELBO equals $\log p(x)$ — the bound touches, as in EM's E-step.</details>
3. In the CAVI update $q(\mu)=\mathcal N(\mu_N,[(\lambda_0+N)\mathbb E\tau]^{-1})$, suppose $N=20$, $\lambda_0=1$ and $\mathbb E[\tau]=2.5$. What is the standard deviation of $q(\mu)$? Is the exact posterior standard deviation larger or smaller?
   <details><summary>Answer</summary>$\lambda_q=21\times2.5=52.5$, so sd $=1/\sqrt{52.5}\approx0.138$. The exact marginal sd is larger, because the exact posterior averages $1/((\lambda_0+N)\tau)$ over the uncertainty in $\tau$ (giving a Student-t), whereas mean-field plugs in $\mathbb E[\tau]$ — and $\mathbb E[1/\tau]>1/\mathbb E[\tau]$ by Jensen's inequality.</details>
4. For $f(z)=z^2$ and $q=\mathcal N(\mu,\sigma^2)$, write the reparameterised gradient estimators for $\mu$ and $\sigma$ and check them against the exact gradients of $\mathbb E_q[z^2]=\mu^2+\sigma^2$.
   <details><summary>Answer</summary>$z=\mu+\sigma\varepsilon$, $f'(z)=2z$. $\nabla_\mu$: $\mathbb E[2(\mu+\sigma\varepsilon)]=2\mu$. $\nabla_\sigma$: $\mathbb E[2(\mu+\sigma\varepsilon)\varepsilon]=2\sigma$ (since $\mathbb E\varepsilon=0$, $\mathbb E\varepsilon^2=1$). Both match $\partial(\mu^2+\sigma^2)/\partial\mu=2\mu$ and $\partial/\partial\sigma=2\sigma$.</details>
5. What exactly does the VAE encoder *amortise*, and what is the benefit for a new batch of cells?
   <details><summary>Answer</summary>It replaces a separate optimisation of $(\mu_c,\sigma_c)$ for every cell by one network that outputs them for any input: the cost of inference is paid once during training. A new cell gets its approximate posterior in a single forward pass, with no per-cell optimisation (the price is an *amortisation gap*: the network's answer may be slightly worse than optimising that cell's $q$ directly).</details>
6. Where does the library size enter the NB decoder, and why not let the network output the mean count directly?
   <details><summary>Answer</summary>$\mu_{cg}=\ell_c\,\rho_{cg}(z_c)$: the decoder outputs proportions (softmax over genes) and the observed library size multiplies them. If the network predicted counts directly, the latent code would have to encode sequencing depth — a technical factor — and cells of the same type with different depths would get different codes.</details>
7. Code reading: in `train_vae`, what happens to the latent space if `kl_weight=20`? Which numbers in Section 8 show it, and why does the reconstruction term hardly suffer?
   <details><summary>Answer</summary>The KL term dominates, so $q(z\mid x)\to\mathcal N(0,I)$ for every cell: posterior collapse. Section 8 shows no active dimension, KL ≈ 0 and kNN accuracy 0.35 (chance ≈ 0.17). The reconstruction term stays reasonable because $\ell_c\times$ (average profile) already explains most of each cell's counts; only cell-type-specific deviations are lost.</details>
8. Why does feeding the batch label to the decoder remove batch effects from $z$, and in which experimental design would it remove real biology instead?
   <details><summary>Answer</summary>The decoder can explain batch shifts from the observed label for free, and the KL term penalises any information in $z$ that the decoder does not need, so $z$ stops encoding batch. If batch is confounded with biology — e.g. all stimulated donors in one batch, or a cell type present in one batch only — the model cannot tell the two apart and attributes the biological difference to batch, erasing it from $z$ (Module 31).</details>

## 🧠 Questions & discussion

1. Section 4 showed mean-field VI declaring narrow intervals for SNPs in LD. In a genome-wide fine-mapping study run with VI to save time, how would you detect and limit this over-confidence before reporting candidate causal variants?
2. EM, Gibbs sampling ([Module 21](21_bayesian_learning_mcmc.ipynb)) and CAVI all update one block of unknowns given the others. What does each one compute for the block, and what does each guarantee?
3. The NB VAE's learned dispersions were far smaller than the simulator's true ones. Give two biological or technical reasons why a model's 'noise' parameter can differ from the true measurement noise, and one consequence for downstream analyses such as differential expression.
4. A VAE trained on healthy donors is used to embed cells from patients, and cells far from all healthy cells are labelled 'disease-specific states'. What could produce such cells besides disease, and what controls would you ask for?
5. Latent spaces of large single-cell models are increasingly used to make claims about cell identity across atlases. What are the risks of treating a latent coordinate system learned from one population (e.g. mostly European-ancestry donors) as a universal reference, and how could they be mitigated?

## 🚀 Bonus — going further

- **β-VAE sweep.** Train the NB VAE for $\beta\in\{0.25,0.5,1,2,5\}$ and plot active dimensions, cell-type silhouette and reconstruction against $\beta$. Where is the best trade-off for this data set?
- **Better posteriors.** Repeat Section 4 with a full-covariance Gaussian $q$ (✏️ in Section 4), then with a normalising flow (a few learned invertible transformations of $\varepsilon$). How close do you get to the MH intervals?
- **Semi-supervised VAE (the scANVI idea).** Add a classifier head on $z$ that predicts the cell type for 5 % of cells with labels, add its cross-entropy to the loss, and measure label-transfer accuracy on the unlabelled 95 %.
- **Estimating $\log p(x)$.** Implement the importance-weighted bound $\log\frac1K\sum_k p(x,z_k)/q(z_k\mid x)$ with $K=50$ samples (Burda *et al.* 2016) and compare it with the ELBO for a few cells: how large is the gap?
- **Try the real thing.** Install scvi-tools in a fresh Colab and train `scvi.model.SCVI` on a public PBMC data set from CELLxGENE or 10x Genomics, with `batch_key` set; compare its latent space with 50 PCs using the metrics of Section 6 and known marker genes.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 25 · Variational inference and variational autoencoders
Biological question: can a probabilistic model of raw counts learn a low-dimensional cell state that
                     separates cell types and removes technical effects (library size, batch)?
Data (simulated): Normal-Gamma gene toy (true μ, τ); simulate_genotypes + logistic disease model (true β,
                  one null SNP in LD); simulate_scrna PBMC-like counts (true types, condition, batch, θ_g)
Method: KL grid search; CAVI (mean-field); reparameterised stochastic VI vs Metropolis-Hastings;
        Gaussian VAE and NB VAE (mini-scVI); batch-conditioned VAE; β-VAE collapse test
Key equation: log p(x) = ELBO(q) + KL(q || p(z|x));  ELBO = E_q[log p(x|z)] − KL(q(z|x) || N(0, I));
              μ_cg = ℓ_c ρ_cg(z_c),  x_cg ~ NB(μ_cg, θ_g)
Key parameters: latent dimension, hidden units, epochs, batch size, learning rate, KL warm-up, KL weight β
Ground-truth check result: (kNN accuracy / silhouette / ARI vs PCA; zero-fraction error; batch entropy;
                           counterfactual correlation; VI vs MH interval widths)
What changed when I modified … (n_latent / β / number of cells / seed): …
One failure mode: (e.g. β = 20 → posterior collapse with a reasonable-looking loss; VI too confident for SNPs in LD)
Connection to research (paper/tool): scVI (Lopez et al. 2018), scvi-tools; Blei et al. 2017 review
```

## 📚 References

- Blei D. M., Kucukelbir A., McAuliffe J. D. (2017) Variational inference: a review for statisticians. *J Am Stat Assoc* 112:859–877.
- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — Chapter 10 (approximate inference; the Normal–Gamma example of §10.1.3).
- Jordan M. I., Ghahramani Z., Jaakkola T. S., Saul L. K. (1999) An introduction to variational methods for graphical models. *Machine Learning* 37:183–233.
- Hoffman M. D., Blei D. M., Wang C., Paisley J. (2013) Stochastic variational inference. *J Mach Learn Res* 14:1303–1347.
- Ranganath R., Gerrish S., Blei D. M. (2014) Black box variational inference. *Proc AISTATS*.
- Williams R. J. (1992) Simple statistical gradient-following algorithms for connectionist reinforcement learning. *Machine Learning* 8:229–256 (the score-function / REINFORCE estimator).
- Kingma D. P., Welling M. (2014) Auto-encoding variational Bayes. *ICLR* (arXiv:1312.6114).
- Rezende D. J., Mohamed S., Wierstra D. (2014) Stochastic backpropagation and approximate inference in deep generative models. *Proc ICML*.
- Kingma D. P., Welling M. (2019) An introduction to variational autoencoders. *Found Trends Mach Learn* 12:307–392.
- Tipping M. E., Bishop C. M. (1999) Probabilistic principal component analysis. *J R Stat Soc B* 61:611–622.
- Bowman S. R., Vilnis L., Vinyals O., Dai A. M., Jozefowicz R., Bengio S. (2016) Generating sentences from a continuous space. *Proc CoNLL* (KL annealing, posterior collapse).
- Higgins I. *et al.* (2017) β-VAE: learning basic visual concepts with a constrained variational framework. *ICLR*.
- Burda Y., Grosse R., Salakhutdinov R. (2016) Importance weighted autoencoders. *ICLR* (active units, importance-weighted bound).
- Lopez R., Regier J., Cole M. B., Jordan M. I., Yosef N. (2018) Deep generative modeling for single-cell transcriptomics. *Nat Methods* 15:1053–1058.
- Gayoso A. *et al.* (2022) A Python library for probabilistic analysis of single-cell omics data. *Nat Biotechnol* 40:163–166.
- Xu C., Lopez R., Mehlman E., Regier J., Jordan M. I., Yosef N. (2021) Probabilistic harmonization and annotation of single-cell transcriptomics data with deep generative models. *Mol Syst Biol* 17:e9620.
- Gayoso A. *et al.* (2021) Joint probabilistic modeling of single-cell multi-omic data with totalVI. *Nat Methods* 18:272–282.
- Lotfollahi M., Wolf F. A., Theis F. J. (2019) scGen predicts single-cell perturbation responses. *Nat Methods* 16:715–721.
- Lotfollahi M. *et al.* (2023) Predicting cellular responses to complex perturbations in high-throughput screens. *Mol Syst Biol* 19:e11517.
- Svensson V. (2020) Droplet scRNA-seq is not zero-inflated. *Nat Biotechnol* 38:147–150.
- Kang H. M. *et al.* (2018) Multiplexed droplet single-cell RNA-sequencing using natural genetic variation. *Nat Biotechnol* 36:89–94.
- Zhong E. D., Bepler T., Berger B., Davis J. H. (2021) CryoDRGN: reconstruction of heterogeneous cryo-EM structures using neural networks. *Nat Methods* 18:176–185.

## ➡️ Next

VAEs compress each cell into a vector; the next building block lets models relate *parts* of an input to each other —
residues in a protein, bases in a promoter, genes in a cell. **[Module 26 · Attention, transformers and
self-supervised learning](26_attention_transformers.ipynb)** builds attention from a learned soft k-NN and trains a tiny
protein language model. If you follow the single-cell track, continue with
**[Module 30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb)** and then
**[Module 31 · Batch effects and integration](31_batch_integration.ipynb)**, which reuses this notebook's batch-aware VAE.